# Final version (v7): language-supervised multimodal experts for predicting BI-RADS ≥ 4 from images only

**Proposed method (one model):**
1. Every MG and US image is encoded by a frozen DINOv2 and summarised by label-dependent attention.
2. The modalities are fused by a **label-dependent mixture of experts** with four experts: MG, US, joint concatenation and joint mask-aware mean.
3. **The report supervises training only**, in four ways:
   - (a) cross-fitted distillation from a report teacher
   - (b) image-report alignment
   - (c) routing guided by the MG and US sections of the report
   - (d) a local LLM (Qwen2.5-7B-Instruct) extracts 15 structured BI-RADS lexicon findings from every masked report, and the image model learns to predict them from the pixels.
4. At test time only the images enter the model.

**Confirmatory evaluation:** all runs use **new outer splits** (seeds 11, 12, 13) that were not used for any design decision. The only comparisons are with the published method on this dataset (Nouman & Haraguchi 2026, re-implemented on the same encoders) and the necessary ablations.

## 0) Libraries and folder structure

Expected structure. Place the notebook **next to** the data folder (or change `DATA_DIR` in the settings cell):
```
project/
├── MM_LDAM_BreastCancer.ipynb
└── Breast Cancer Project/
    ├── BCMID_annotations.tsv      (or an Excel .xlsx file; both are supported)
    ├── 0103y17/
    │   ├── Mammogram/0103y17_L_CC_3.png, 0103y17_L_MLO_2.png, ...
    │   ├── Ultrasound/0103y17_US_L_1.png, ...
    │   └── 0103y17_US_MG_Report.pdf
    └── ...
```
The first run downloads the pretrained weights from Hugging Face (Bio_ClinicalBERT about 440 MB, DINOv2-Base about 350 MB).

In [ ]:
# Installs any missing package into THIS kernel's Python (works in Anaconda / Windows too).
# torch is not installed automatically: for a GPU install the CUDA build from https://pytorch.org
import importlib, importlib.util, subprocess, sys
REQUIRED = {"timm": "timm", "transformers": "transformers", "pypdf": "pypdf", "sklearn": "scikit-learn",
            "scipy": "scipy", "pandas": "pandas", "openpyxl": "openpyxl", "matplotlib": "matplotlib", "tqdm": "tqdm"}
missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    importlib.invalidate_caches()
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing: see https://pytorch.org/get-started/locally/")
print("all required packages are available")

In [ ]:
import os, re, math, json, glob, random, hashlib, warnings, time
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from scipy import ndimage as ndi
from tqdm.auto import tqdm
from IPython.display import HTML, display

import torch
import torch.nn as nn
import torch.nn.functional as TF

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             confusion_matrix, f1_score, balanced_accuracy_score)

warnings.filterwarnings("ignore", category=UserWarning)
Image.MAX_IMAGE_PIXELS = None          # full-field mammograms are ~28 MP
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True  # accept slightly truncated PNG/JPEG files

DEVICE = ("cuda" if torch.cuda.is_available()
          else "mps" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
          else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

## 1) Settings
All important parameters are here. Changing feature-extraction parameters (models, image size, `TEXT_MODE`) builds a new cache; training parameters do not.

In [ ]:
# ---------------- paths ----------------
def _find_data_dir():
    for cand in [Path("Breast cancer"), Path("Breast_cancer"), Path("data"), Path(".")]:
        if cand.is_dir() and any(cand.glob("*.tsv")) | any(cand.glob("*.xlsx")) | any(cand.glob("*.xls")) | any(cand.glob("*.csv")):
            return cand
    return Path(".")

DATA_DIR   = _find_data_dir()     # folder with patient sub-folders + label file
LABEL_FILE = None                 # None = auto-detect (*.xlsx / *.xls / *.csv / *.tsv inside DATA_DIR)
CACHE_DIR  = Path("cache_mmldam") # extracted features are stored here (safe to delete)
OUT_DIR    = Path("outputs_mmldam")

# ---------------- pretrained encoders (frozen) ----------------
TEXT_MODEL     = "emilyalsentzer/Bio_ClinicalBERT"    # the Clinical-BERT used by the LDAM paper [ref 2]
IMG_BACKBONE   = "vit_base_patch14_dinov2.lvd142m"    # any timm model; e.g. "convnext_small.fb_in22k_ft_in1k"
IMG_PRETRAINED = True
MG_SIZE = (518, 406)          # (H, W) mammogram input after breast crop, multiple of 14 for DINOv2
US_SIZE = (392, 518)          # (H, W) ultrasound input after scan-area crop
GRID_MG = (8, 6)              # patch-token grid kept per mammogram (adaptive average pooling)
GRID_US = (6, 8)              # patch-token grid kept per ultrasound image
MAX_MG_IMAGES = 8
MAX_US_IMAGES = 12            # if a patient has more, evenly spaced images are kept
MAX_TEXT_TOKENS = 1024        # long reports are encoded in 512-token chunks

# ---------------- labels / leakage control ----------------
TEXT_MODE    = "masked"       # "masked" | "findings_only" | "raw" (raw = leaks BI-RADS, only for comparison)
DROP_BIRADS6 = False          # True = drop already biopsy-proven cancers (BI-RADS 6) from the study
LABELS = ["Normal", "Benign", "Malignant"]            # multi-label head as in the paper; cancer = "Malignant"
LABEL_TEXTS = {                                        # label *names* embedded by Clinical-BERT (E_Y)
    "Normal":    "normal breast with no abnormality, negative for cancer",
    "Benign":    "benign breast lesion, negative for cancer",
    "Malignant": "malignant breast cancer, carcinoma, positive for cancer",
}
CANCER_IDX = LABELS.index("Malignant")

# ---------------- model / training ----------------
F_DIM        = 256     # size of z vectors (F in the paper)
TEXT_NGRAM   = 3       # k1 of the 1D-CNN on the cross-attention matrix (text)
CTX_LAYERS   = 1       # Transformer layers over image tokens (paper: BiGRU); 0 = off
DROPOUT      = 0.3
MOD_DROPOUT  = 0.2     # probability to hide a whole modality during training (robust to missing data)
IMG_DROPOUT  = 0.15    # probability to drop individual images during training
TOKEN_DROPOUT= 0.1     # probability to drop individual tokens (words / patches) during training
AUX_W        = 0.3     # weight of per-modality auxiliary heads
LABEL_REG_W  = 0.1     # weight of the label-embedding term of Eq. (12)
LR, WEIGHT_DECAY = 3e-4, 1e-2
EPOCHS, PATIENCE, BATCH_SIZE = 60, 12, 16
N_FOLDS, SEED = 5, 42
RUN_ABLATIONS = False # True

CACHE_DIR.mkdir(parents=True, exist_ok=True); OUT_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
seed_everything()
print("DATA_DIR =", DATA_DIR.resolve())

In [ ]:
# ==================== v2/v3 settings ====================
# Main task: image-only prediction of BI-RADS >= 4. The report is used ONLY during training (privileged information).
N_SEEDS       = 3        # members per outer fold (different seed AND inner-validation split); predictions are averaged
CALIBRATE     = True     # Platt scaling of each member on its own inner-validation set before averaging
SPEC_TARGET   = 0.90     # sensitivity is also reported at 90 % specificity
N_BOOT        = 2000     # bootstrap resamples for confidence intervals

TEACHER_USE   = ("text",)  # teachers read only the (masked) report or one of its sections
TEACHER_FOLDS = 3          # cross-fitting: the teacher target of every training patient is out-of-fold
KD_W, KD_T    = 1.0, 2.0   # distillation weight / temperature     (fixed a priori - NOT tuned on test folds)
ALIGN_W       = 0.1        # image-report contrastive weight       (fixed a priori)
ALIGN_DIM     = 128
GATE_W, GATE_T = 0.5, 1.0  # report-guided routing weight / temperature (fixed a priori)

RUN_ALL_COHORT = True     # secondary analysis on all patients (incl. BI-RADS 6); the primary cohort is BI-RADS 1-5
ORD_W, N_ORD   = 0.5, 5   # ordinal BI-RADS auxiliary head: P(BI-RADS > k), k = 1..5 (candidate, chosen on inner validation)
REPEAT_SEEDS   = (1, 2)   # repeat MM-LDAM / mask-aware fusion / final with two other outer-fold splits (confirmation)
LOO_ALL_COHORT = True     # leave-one-component-out ablation also on all patients
SHORTCUT_PID   = "00263y23"   # BI-RADS 6 case whose US attention fell on a biopsy-clip annotation (shortcut example)
PUBLISHED = {"Nouman and Haraguchi 2026": 0.871}   # reported positive-class AUROC on all 332 cases (different protocol)
FORCE_RERUN   = False    # False = reuse saved out-of-fold predictions of experiments that already finished
MOD_NAME      = {"text": "Report", "mg": "MG", "us": "US"}
FIG_DIR = OUT_DIR / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
(OUT_DIR / "oof").mkdir(parents=True, exist_ok=True)
# ---------------- v7: unified method ----------------
ATTR_W      = 0.3                           # weight of the LLM-findings head (training only)
USE_LLM     = True                          # False = rule-based extraction of the same findings
LLM_MODEL   = "Qwen/Qwen2.5-7B-Instruct"    # open model, runs locally; smaller GPU: "Qwen/Qwen2.5-3B-Instruct"
LLM_4BIT    = True # False                         # True = 4-bit loading (needs bitsandbytes), for GPUs with < 16 GB
REPEATS_P   = (11, 12, 13)                  # fresh outer splits, primary cohort (BI-RADS 1-5); add 14, 15 if time allows
REPEATS_ALL = (11,)                         # outer split(s) for the all-patient cohort
print("settings loaded (v7)")

## 2) Labels and data checks
- IDs are read as text so that leading zeros (e.g. `00263y23`) are kept.
- BI-RADS 0 rows (incomplete assessment) have no binary label and are left out.
- **Patient group** = the part of the ID before `y` (e.g. `1000y16` and `1000y17` → group `1000`).

In [ ]:
def find_label_file(data_dir):
    if LABEL_FILE is not None:
        return Path(LABEL_FILE)
    cands = [p for ext in ("*.xlsx", "*.xls", "*.csv", "*.tsv") for p in Path(data_dir).glob(ext)]
    if not cands:
        raise FileNotFoundError(f"No label file (*.xlsx/*.csv/*.tsv) found in {data_dir}")
    return cands[0]

def read_labels(path):
    path = Path(path)
    if path.suffix.lower() in (".xlsx", ".xls"):
        df = pd.read_excel(path, dtype=str)
    else:
        df = pd.read_csv(path, sep="\t" if path.suffix.lower() == ".tsv" else None, dtype=str, engine="python")
    df.columns = [c.strip() for c in df.columns]
    cols = list(df.columns)
    def pick(keys, default_pos):
        for c in cols:
            if any(k in c.lower() for k in keys):
                return c
        return cols[default_pos]
    c_id, c_bi = pick(["id"], 0), pick(["bi-rads", "birads", "bi_rads"], 1)
    c_bin, c_pat = pick(["binary"], 2), pick(["patholog"], 3)
    out = pd.DataFrame({
        "pid":       df[c_id].astype(str).str.strip(),
        "birads":    pd.to_numeric(df[c_bi], errors="coerce"),
        "binary":    df[c_bin].astype(str).str.strip().str.capitalize().replace({"Nan": np.nan, "": np.nan}),
        "pathology": df[c_pat].astype(str).str.strip().str.capitalize().replace({"Nan": np.nan, "": np.nan}),
    })
    return out

labels_df = read_labels(find_label_file(DATA_DIR))
print("rows:", len(labels_df))
display(pd.crosstab(labels_df.birads, labels_df.pathology.fillna("—"), margins=True))

# keep rows with a usable label
labels_df = labels_df[labels_df.binary.isin(["Positive", "Negative"])].copy()
if DROP_BIRADS6:
    labels_df = labels_df[labels_df.birads != 6].copy()
labels_df["y"] = (labels_df.binary == "Positive").astype(int)
# fill pathology from binary when missing
labels_df.loc[labels_df.pathology.isna() & (labels_df.y == 1), "pathology"] = "Malignant"
labels_df.loc[labels_df.pathology.isna() & (labels_df.y == 0), "pathology"] = "Benign"
labels_df["group"] = labels_df.pid.str.split("y").str[0]
labels_df = labels_df.drop_duplicates("pid").reset_index(drop=True)
print(f"labelled patients: {len(labels_df)} | cancer-positive: {labels_df.y.sum()} "
      f"({labels_df.y.mean():.1%}) | patient groups: {labels_df.group.nunique()}")

In [ ]:
IMG_EXT = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".dcm")
TXT_EXT = (".pdf", ".txt", ".docx")

def index_patient_folders(data_dir):
    """Map folder name -> path (searching two levels deep, case-insensitive)."""
    idx = {}
    for p in Path(data_dir).iterdir():
        if p.is_dir():
            idx[p.name.lower()] = p
            for q in p.iterdir():
                if q.is_dir() and q.name.lower() not in idx and re.search(r"y\d", q.name):
                    idx[q.name.lower()] = q
    return idx

def list_patient_files(folder):
    folder = Path(folder)
    files = [p for p in folder.rglob("*") if p.is_file()]
    mg = sorted([p for p in files if p.suffix.lower() in IMG_EXT and
                 ("mammo" in str(p.parent).lower() or re.search(r"_(CC|MLO)_", p.name, re.I))])
    us = sorted([p for p in files if p.suffix.lower() in IMG_EXT and p not in mg and
                 ("ultra" in str(p.parent).lower() or "_US_" in p.name.upper())])
    rep = sorted([p for p in files if p.suffix.lower() in TXT_EXT])
    return mg, us, rep

folder_index = index_patient_folders(DATA_DIR)
labels_df["folder"] = labels_df.pid.str.lower().map(folder_index)
audit = []
for r in labels_df.itertuples():
    if r.folder is None or (isinstance(r.folder, float) and np.isnan(r.folder)):
        audit.append((0, 0, 0)); continue
    mg, us, rep = list_patient_files(r.folder)
    audit.append((len(mg), len(us), len(rep)))
labels_df[["n_mg", "n_us", "n_report"]] = audit
missing = labels_df.folder.isna().sum()
print(f"patients with a folder: {len(labels_df) - missing} / {len(labels_df)}")
if missing:
    print("  (patients without a folder are skipped:", ", ".join(labels_df.pid[labels_df.folder.isna()].head(10)), "...)")
labels_df = labels_df[labels_df.folder.notna()].reset_index(drop=True)
display(labels_df[["n_mg", "n_us", "n_report"]].describe().T[["mean", "min", "max"]])
display(labels_df.head())

## 3) Report text preprocessing (with label-leakage control)

In [ ]:
def _pdf_pymupdf(path):
    try:
        import pymupdf as fitz
    except ImportError:
        import fitz
    with fitz.open(str(path)) as doc:
        return "\n".join(p.get_text("text") for p in doc)

def _pdf_pypdf(path):
    from pypdf import PdfReader
    return "\n".join((p.extract_text() or "") for p in PdfReader(str(path)).pages)

def _pdf_pdfplumber(path):
    import pdfplumber
    with pdfplumber.open(str(path)) as pdf:
        return "\n".join((p.extract_text() or "") for p in pdf.pages)

def _pdf_pdfminer(path):
    from pdfminer.high_level import extract_text
    return extract_text(str(path))

PDF_BACKENDS = [_pdf_pymupdf, _pdf_pypdf, _pdf_pdfplumber, _pdf_pdfminer]   # first one that works is used
REPORT_ERRORS = {}

def read_report(path):
    """Text of a report (.pdf / .docx / .txt). Tries several PDF libraries; '' if nothing works."""
    path = Path(path)
    ext = path.suffix.lower()
    try:
        if ext == ".docx":
            import docx
            return "\n".join(par.text for par in docx.Document(str(path)).paragraphs)
        if ext != ".pdf":
            return path.read_text(errors="ignore")
    except Exception as e:
        REPORT_ERRORS[str(path)] = repr(e); return ""
    errs = []
    for fn in PDF_BACKENDS:
        try:
            txt = fn(path)
            if txt and txt.strip():
                return txt
            errs.append(f"{fn.__name__}: empty text")
        except Exception as e:
            errs.append(f"{fn.__name__}: {type(e).__name__}: {e}")
    REPORT_ERRORS[str(path)] = " | ".join(errs)
    return ""

_LEGEND  = re.compile(r"Category\s+Breast\s+imaging\s+reporting.*", re.I | re.S)
# no leading \b on purpose: some PDF libraries glue words together ("follow upBIRADS 2Category ...")
_BIRADS  = re.compile(r"(?:B\W{0,2}I\W{0,2}-?\W{0,2}RADS?|BIRADS?|BI\W{0,2}RAD|B\W{0,2}IRAD)\W{0,4}(?:category\W*)?\(?\s*[0-6](?:\s*[abc](?![a-z]))?\s*\)?", re.I)
_BIWORD  = re.compile(r"(?:B\W{0,2}I\W{0,2}-?\W{0,2}RADS?|BIRADS?|B\W{0,2}IRAD)", re.I)
_DOCTOR  = re.compile(r"(?:Dr|DR)\s*\.\s*[A-Za-z][\w\.]*(?:\s+[A-Za-z][\w\.]*){0,3}(?:\s*,\s*M\.?\s*S\.?c?\.?)?")   # case-sensitive on purpose
_ADMIN   = re.compile(r"|".join([                     # administrative header fields (IDs, referring doctor, dates)
    r"\bID\s*:\s*[\d/]+",
    r"Ref\.?\s*(?:Dr|Prof)\.?\s*:?.*?(?=procedure|study\s*date|clinical\s*data|age\s*:|\n|$)",
    r"Study\s*Date\s*:\s*[\w/\-\.]+(?:\s+[\d:]+)?(?:\s*[AP]M)?"]), re.I)
_LEXICON = re.compile("|".join([          # BI-RADS assessment-category wording (= the label itself)
    r"incomplete assessment", r"negative findings?", r"normal study", r"benign findings?", r"probably benign",
    r"suspicious abnormality", r"biopsy is recommended", r"highly suggestive of malignan\w*",
    r"proved malign\w* by \w+", r"proven malign\w*", r"(?:small|moderate|substantial) risk",
    r"\d+\s*months? follow\s*-?\s*up", r"annual follow\s*-?\s*up"]), re.I)
_OPINION = re.compile(r"(?:OPINION|IMPRESSION|CONCLUSION)S?\b\s*:?.*", re.I | re.S)   # radiologist's conclusion
_PROCEDURE = re.compile(r"procedure\s*:.*?(?=study\s*date|clinical\s*data|\n|$)", re.I)       # e.g. "clip insertion" (= known cancer)

def clean_report(raw, mode=TEXT_MODE):
    t = raw.replace("…", " ").replace("’", "'")
    if mode != "raw":
        t = _LEGEND.sub(" ", t)
        if mode == "findings_only":
            t = _OPINION.sub(" ", t); t = _PROCEDURE.sub(" ", t)
        t = _BIRADS.sub(" ", t); t = _BIWORD.sub(" ", t); t = _LEXICON.sub(" ", t)
        t = _ADMIN.sub(" ", t)
        t = _DOCTOR.sub(" ", t)          # radiologist names can act as a shortcut (who reads which cases)
    t = re.sub(r"[\.…]{2,}", " . ", t)
    t = re.sub(r"[^\w\s\.,;:%/\-\(\)'x]", " ", t)
    return re.sub(r"\s+", " ", t).strip().lower()

def patient_report_text(folder, mode=TEXT_MODE):
    _, _, reps = list_patient_files(folder)
    return " ".join(clean_report(read_report(p), mode) for p in reps).strip()

ex = labels_df.iloc[0]
_, _, _reps = list_patient_files(ex.folder)
if _reps:
    raw = read_report(_reps[0])
    print("RAW  :", re.sub(r"\s+", " ", raw)[:700], "\n")
    print("CLEAN:", clean_report(raw)[:700])
# read every report once: (1) were they readable?  (2) does any BI-RADS mention survive the masking?
REPORT_TEXT = {r.pid: patient_report_text(r.folder) for r in tqdm(labels_df.itertuples(), total=len(labels_df), desc="reports")}
has_file = labels_df.set_index("pid").n_report > 0
empty = [p for p, t in REPORT_TEXT.items() if has_file[p] and not t]
print(f"\nreports read: {sum(bool(t) for t in REPORT_TEXT.values())} | report file present but no text: {len(empty)}")
if REPORT_ERRORS:
    k, v = next(iter(REPORT_ERRORS.items())); print("example error ->", k, "\n   ", v)
if len(empty) > 0.2 * max(has_file.sum(), 1):
    raise RuntimeError("Most reports could not be read. Install a PDF library in this kernel, e.g. run "
                       "`%pip install pypdf` in a cell, restart the kernel, and run the notebook again. "
                       "If the PDFs are scanned images (no text layer) they need OCR first.")
_left = [p for p, t in REPORT_TEXT.items() if re.search(r"b\W?i\W?-?\W?rads?\W{0,3}[0-6]", t)]
print("reports still containing a BI-RADS score:", _left[:10] if _left else "none ✓")

## 4) Image preprocessing
**Mammography:** (1) polarity correction: some images are stored as negatives (white background) and are inverted; (2) breast segmentation with the largest connected component (removes labels such as `L` and `R-CC`); (3) crop to the breast; (4) orientation (chest wall on the left); (5) intensity normalisation and resizing with preserved aspect ratio.

**Ultrasound:** (1) removal of coloured caliper markers; (2) crop to the scan area (removes scanner panels, depth numbers and the `Dist A` measurement box); (3) normalisation and resizing. Removing calipers matters because a measurement on the image is itself a cue that the finding is suspicious.

In [ ]:
def _read_dicom(path):
    import pydicom
    ds = pydicom.dcmread(str(path), force=True); a = ds.pixel_array.astype(np.float32)
    if getattr(ds, "PhotometricInterpretation", "") == "MONOCHROME1":
        a = a.max() - a
    return a

def read_image_array(path):
    """Returns a float32 array (H,W) or (H,W,3). Works even if the extension is wrong
    (e.g. a JPEG or DICOM saved as .png). Raises IOError with a diagnostic if nothing can read it."""
    path = Path(path); errs = []
    if path.suffix.lower() == ".dcm":
        return _read_dicom(path)
    try:
        with Image.open(path) as im:
            im.load()
            if im.mode in ("I;16", "I;16B", "I", "F"):
                return np.asarray(im, np.float32)
            return np.asarray(im.convert("RGB" if im.mode in ("RGB", "RGBA", "P", "CMYK") else "L"), np.float32)
    except Exception as e:
        errs.append(f"PIL: {type(e).__name__}")
    try:
        return _read_dicom(path)
    except Exception as e:
        errs.append(f"pydicom: {type(e).__name__}")
    try:
        import cv2
        a = cv2.imdecode(np.fromfile(str(path), np.uint8), cv2.IMREAD_UNCHANGED)
        if a is not None:
            if a.ndim == 3: a = cv2.cvtColor(a[..., :3], cv2.COLOR_BGR2RGB)
            return a.astype(np.float32)
        errs.append("cv2: None")
    except Exception as e:
        errs.append(f"cv2: {type(e).__name__}")
    size = path.stat().st_size if path.exists() else -1
    head = path.read_bytes()[:8] if size > 0 else b""
    raise IOError(f"unreadable image ({size} bytes, header {head!r}; {'; '.join(errs)})")

def _unit(a):
    a = a - a.min()
    return a / max(float(a.max()), 1e-6)

def load_gray(path):
    a = read_image_array(path)
    return _unit(a.mean(-1) if a.ndim == 3 else a)

def load_rgb(path):
    a = read_image_array(path)
    return np.stack([_unit(a)] * 3, -1) if a.ndim == 2 else _unit(a[..., :3])

def _otsu(x):
    hist, edges = np.histogram(x, bins=256, range=(0, 1))
    p = hist / max(hist.sum(), 1); w = np.cumsum(p); mu = np.cumsum(p * np.arange(256))
    sb = (mu[-1] * w - mu) ** 2 / (w * (1 - w) + 1e-12)
    return edges[np.argmax(sb)]

def _resize(a, hw):
    return np.asarray(Image.fromarray((np.clip(a, 0, 1) * 255).astype(np.uint8)).resize((hw[1], hw[0]), Image.BILINEAR), np.float32) / 255

def letterbox(a, out_hw):
    H, W = out_hw; h, w = a.shape; s = min(H / h, W / w)
    nh, nw = max(1, round(h * s)), max(1, round(w * s))
    out = np.zeros((H, W), np.float32); out[:nh, :nw] = _resize(a, (nh, nw))
    return out

def _largest_component(m):
    lab, n = ndi.label(m)
    if n == 0:
        return m, 0.0
    sizes = ndi.sum(m, lab, range(1, n + 1)); k = int(np.argmax(sizes)) + 1
    return lab == k, sizes[k - 1] / m.size

def preprocess_mammogram(path, out_hw=MG_SIZE):
    a = load_gray(path)
    s = max(a.shape) / 2048
    if s > 1:
        a = _resize(a, (int(a.shape[0] / s), int(a.shape[1] / s)))
    if (a > 0.85).mean() > (a < 0.15).mean():          # stored as negative -> invert
        a = 1.0 - a
    b = max(2, int(0.015 * min(a.shape)))               # scanner edge strips
    a[:b] = 0; a[-b:] = 0; a[:, :b] = 0; a[:, -b:] = 0
    sm = ndi.gaussian_filter(a, 3)
    m = ndi.binary_opening(sm > max(_otsu(sm) * 0.5, 0.04), iterations=3)
    m, frac = _largest_component(m)
    if frac > 0.05:
        m = ndi.binary_fill_holes(m); ys, xs = np.where(m)
        a = (a * m)[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
        m = m[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    else:
        m = np.ones_like(a, bool)
    if a[:, : a.shape[1] // 2].sum() < a[:, a.shape[1] // 2:].sum():   # chest wall to the left
        a, m = a[:, ::-1], m[:, ::-1]
    lo, hi = np.percentile(a[m], [0.5, 99.5])
    return letterbox(np.clip((a - lo) / max(hi - lo, 1e-6), 0, 1) * m, out_hw)

def preprocess_ultrasound(path, out_hw=US_SIZE):
    rgb = load_rgb(path)
    g = rgb.mean(-1)
    col = ndi.binary_dilation((rgb.max(-1) - rgb.min(-1)) > 0.15, iterations=2)   # coloured calipers
    if col.any():
        g = np.where(col, ndi.median_filter(g, size=9), g)
    m = ndi.binary_opening(ndi.gaussian_filter(g, 2) > 0.06, iterations=4)
    m, frac = _largest_component(m)
    if frac > 0.08:
        ys, xs = np.where(m); g = g[ys.min():ys.max() + 1, xs.min():xs.max() + 1]
    lo, hi = np.percentile(g, [0.5, 99.5])
    return letterbox(np.clip((g - lo) / max(hi - lo, 1e-6), 0, 1), out_hw)

def parse_view(path, kind):
    """-> human-readable view name that is embedded with Clinical-BERT (the 'feature name' of the paper)."""
    n = Path(path).stem.upper()
    side = "left" if re.search(r"_L(_|$)", n) else "right" if re.search(r"_R(_|$)", n) else "unknown side"
    if kind == "mg":
        v = "craniocaudal (CC)" if "_CC" in n else "mediolateral oblique (MLO)" if "_MLO" in n else "other"
        return f"{side} breast {v} mammogram view"
    return f"{side} breast ultrasound image"

VIEW_NAMES = ([f"{s} breast {v} mammogram view" for s in ("left", "right", "unknown side")
               for v in ("craniocaudal (CC)", "mediolateral oblique (MLO)", "other")] +
              [f"{s} breast ultrasound image" for s in ("left", "right", "unknown side")])
VIEW2IDX = {v: i for i, v in enumerate(VIEW_NAMES)}

def subsample(lst, k):
    if len(lst) <= k: return list(lst)
    return [lst[i] for i in np.linspace(0, len(lst) - 1, k).round().astype(int)]

# visual check on one malignant and one benign patient
fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for row, lab in enumerate([1, 0]):
    sub = labels_df[labels_df.y == lab]
    for ax in axes[row]: ax.axis("off")
    if len(sub) == 0: continue
    r = sub.iloc[0]; mg, us, _ = list_patient_files(r.folder)
    ims = []
    for p, prep in [(p, preprocess_mammogram) for p in mg[:3]] + [(p, preprocess_ultrasound) for p in us[:3]]:
        try: ims.append((prep(p), p))
        except Exception as e: print("skipped", p, e)
    for ax, (im, p) in zip(axes[row], ims):
        ax.imshow(im, cmap="gray"); ax.set_title(f"{r.pid} ({r.pathology})\n{Path(p).name}", fontsize=8)
plt.tight_layout(); plt.show()

## 5) Feature extraction with frozen encoders (cached)
- **Text:** token embeddings from the last layer of Bio_ClinicalBERT (E_M in the LDAM paper).
- **Label names and view names:** the `[CLS]` vector of the same model (E_Y and E_S in the paper).
- **Images:** DINOv2 patch tokens, average-pooled to a small grid (e.g. 8×6) to keep spatial information at low memory cost.

This step runs once (a few minutes on a GPU, up to about an hour on a CPU). If interrupted, it resumes where it stopped.

In [ ]:
from transformers import AutoTokenizer, AutoModel
from transformers.utils import logging as hf_logging
import timm
hf_logging.set_verbosity_error()      # silences the harmless '>512 tokens' warning (long reports are chunked)

def cfg_hash(d):
    return hashlib.md5(json.dumps(d, sort_keys=True, default=str).encode()).hexdigest()[:10]

TEXT_CACHE = CACHE_DIR / f"text_{cfg_hash([TEXT_MODEL, TEXT_MODE, MAX_TEXT_TOKENS, 'v2'])}"
IMG_CACHE  = CACHE_DIR / f"img_{cfg_hash([IMG_BACKBONE, IMG_PRETRAINED, MG_SIZE, US_SIZE, GRID_MG, GRID_US, MAX_MG_IMAGES, MAX_US_IMAGES])}"
TEXT_CACHE.mkdir(parents=True, exist_ok=True); IMG_CACHE.mkdir(parents=True, exist_ok=True)

_text_enc = {}
def get_text_encoder():
    if not _text_enc:
        tok = AutoTokenizer.from_pretrained(TEXT_MODEL)
        mdl = AutoModel.from_pretrained(TEXT_MODEL).to(DEVICE).eval()
        _text_enc.update(tok=tok, model=mdl)
    return _text_enc["tok"], _text_enc["model"]

@torch.no_grad()
def embed_cls(sentences):
    tok, mdl = get_text_encoder()
    b = tok(sentences, padding=True, truncation=True, max_length=64, return_tensors="pt").to(DEVICE)
    return mdl(**b).last_hidden_state[:, 0].float().cpu()

@torch.no_grad()
def embed_tokens(text):
    """-> (L, D) token embeddings (no [CLS]/[SEP]) and the word-piece strings."""
    tok, mdl = get_text_encoder()
    D = mdl.config.hidden_size
    ids = tok(text, add_special_tokens=False)["input_ids"][:MAX_TEXT_TOKENS] if text else []
    if not ids:
        return torch.zeros(0, D, dtype=torch.float16), []
    embs, maxlen = [], tok.model_max_length if tok.model_max_length < 10000 else 512
    step = maxlen - 2
    for i in range(0, len(ids), step):
        chunk = [tok.cls_token_id] + ids[i:i + step] + [tok.sep_token_id]
        x = torch.tensor([chunk], device=DEVICE)
        embs.append(mdl(input_ids=x, attention_mask=torch.ones_like(x)).last_hidden_state[0, 1:-1].float().cpu())
    return torch.cat(embs).half(), tok.convert_ids_to_tokens(ids)

_img_enc = {}
def get_image_encoder():
    if not _img_enc:
        try:
            m = timm.create_model(IMG_BACKBONE, pretrained=IMG_PRETRAINED, num_classes=0, dynamic_img_size=True)
        except TypeError:
            m = timm.create_model(IMG_BACKBONE, pretrained=IMG_PRETRAINED, num_classes=0)
        cfg = m.pretrained_cfg if hasattr(m, "pretrained_cfg") else {}
        _img_enc.update(model=m.to(DEVICE).eval(),
                        mean=torch.tensor(cfg.get("mean", (0.485, 0.456, 0.406))).view(1, 3, 1, 1),
                        std=torch.tensor(cfg.get("std", (0.229, 0.224, 0.225))).view(1, 3, 1, 1))
    return _img_enc

@torch.no_grad()
def image_tokens(arrs, grid):
    """list of HxW arrays in [0,1] (same size) -> (N, gh*gw, C) patch tokens."""
    enc = get_image_encoder(); m = enc["model"]
    x = torch.from_numpy(np.stack(arrs))[:, None].repeat(1, 3, 1, 1)
    x = ((x - enc["mean"]) / enc["std"]).to(DEVICE)
    with torch.autocast("cuda", enabled=(DEVICE == "cuda")):
        f = m.forward_features(x)
    f = f.float()
    if f.ndim == 3:                                        # ViT: (N, prefix + h*w, C)
        npfx = getattr(m, "num_prefix_tokens", 1)
        ps = m.patch_embed.patch_size; ps = ps[0] if isinstance(ps, (tuple, list)) else ps
        h, w = x.shape[2] // ps, x.shape[3] // ps
        f = f[:, npfx:npfx + h * w].transpose(1, 2).reshape(len(arrs), -1, h, w)
    f = TF.adaptive_avg_pool2d(f, grid)                    # (N, C, gh, gw)
    return f.flatten(2).transpose(1, 2).half().cpu()

BAD_IMAGES = {}

def extract_patient(pid, folder):
    """Compute (or load) cached features for one patient -> dict."""
    tpath, ipath = TEXT_CACHE / f"{pid}.pt", IMG_CACHE / f"{pid}.pt"
    if tpath.exists():
        tfeat = torch.load(tpath)
    else:
        text = patient_report_text(folder)
        emb, toks = embed_tokens(text)
        tfeat = {"text": emb, "tokens": toks, "clean_text": text}
        if text or not list_patient_files(folder)[2]:      # never cache a failed report read
            torch.save(tfeat, tpath)
    if ipath.exists():
        ifeat = torch.load(ipath)
    else:
        mg, us, _ = list_patient_files(folder)
        ifeat = {}
        for key, files, prep, grid, cap in (("mg", mg, preprocess_mammogram, GRID_MG, MAX_MG_IMAGES),
                                            ("us", us, preprocess_ultrasound, GRID_US, MAX_US_IMAGES)):
            first = subsample(files, cap)                   # evenly spaced images; others are back-ups
            done = {}
            for p in first + [q for q in files if q not in first]:
                if len(done) >= cap: break
                try:
                    done[p] = prep(p)
                except Exception as e:                      # corrupt / unreadable file -> skip it
                    BAD_IMAGES[str(p)] = str(e)
                    print(f"  skipped {p}: {e}")
            ok = [p for p in files if p in done]; arrs = [done[p] for p in ok]
            ifeat[f"{key}_files"] = [str(p) for p in ok]
            ifeat[f"{key}_view"] = [VIEW2IDX[parse_view(p, key)] for p in ok]
            ifeat[key] = image_tokens(arrs, grid) if arrs else None
        torch.save(ifeat, ipath)
    return {**tfeat, **ifeat}

# label-name (E_Y) and view-name (E_S) embeddings
emb_path = CACHE_DIR / f"names_{cfg_hash([TEXT_MODEL, LABEL_TEXTS, VIEW_NAMES])}.pt"
if emb_path.exists():
    E_Y, E_VIEW = torch.load(emb_path)
else:
    E_Y = embed_cls([LABEL_TEXTS[l] for l in LABELS]); E_VIEW = embed_cls(VIEW_NAMES)
    torch.save((E_Y, E_VIEW), emb_path)
print("E_Y", tuple(E_Y.shape), "| E_VIEW", tuple(E_VIEW.shape))

FEATS = {}
t0 = time.time()
for r in tqdm(labels_df.itertuples(), total=len(labels_df), desc="feature extraction"):
    FEATS[r.pid] = extract_patient(r.pid, r.folder)
print(f"done in {time.time() - t0:.0f}s")
if BAD_IMAGES:
    print(f"⚠️ {len(BAD_IMAGES)} unreadable image(s) were skipped (listed in {OUT_DIR / 'unreadable_images.csv'})")
    pd.Series(BAD_IMAGES, name="error").to_csv(OUT_DIR / "unreadable_images.csv")
_any = next(iter(FEATS.values()))
TXT_DIM = E_Y.shape[1]
IMG_DIM = next((v[k].shape[-1] for v in FEATS.values() for k in ("mg", "us") if v[k] is not None), 768)
print("text dim", TXT_DIM, "| image token dim", IMG_DIM)

## 6) Dataset and batch construction
Patients have different numbers of text tokens, views and images; all are put in one batch with **padding + masks**. Augmentation happens at the feature level: random token dropout, random image dropout and hiding a whole modality.

In [ ]:
MODALITIES = ("text", "mg", "us")

def make_records(df):
    recs = []
    for r in df.itertuples():
        f = FEATS[r.pid]
        tgt = np.zeros(len(LABELS), np.float32); tgt[LABELS.index(r.pathology)] = 1.0
        t = f.get("text")
        tmean = t.float().mean(0) if (t is not None and len(t) > 0) else None   # report embedding, used ONLY as a training target
        recs.append({"pid": r.pid, "y": int(r.y), "target": tgt, **f, "text_mean": tmean, "birads": float(r.birads),
                     "attr": globals().get("ATTR", {}).get(r.pid)})            # LLM findings, training target only
    return recs

def _img_tokens(rec, key, train):
    feats, views = rec.get(key), rec.get(f"{key}_view", [])
    if feats is None or len(feats) == 0:
        return None
    keep = list(range(len(feats)))
    if train and len(keep) > 1:
        keep = [i for i in keep if random.random() > IMG_DROPOUT] or [random.choice(keep)]
    P = feats.shape[1]
    tok = feats[keep].reshape(-1, feats.shape[-1])
    view = torch.tensor([views[i] for i in keep]).repeat_interleave(P)
    pos = torch.arange(P).repeat(len(keep))
    img = torch.tensor(keep).repeat_interleave(P)
    return tok, view, pos, img

def collate(recs, use=MODALITIES, train=False):
    B = len(recs)
    out = {"target": torch.tensor(np.stack([r["target"] for r in recs])),
           "y": torch.tensor([r["y"] for r in recs])}
    # training-only targets (never a model input): report embedding, teacher soft targets, routing, ordinal and LLM findings
    out["txt_mean"] = torch.stack([r["text_mean"] if r.get("text_mean") is not None else torch.zeros(TXT_DIM) for r in recs])
    out["txt_mean_mask"] = torch.tensor([r.get("text_mean") is not None for r in recs], dtype=torch.bool)
    out["soft"] = torch.tensor(np.stack([r["soft"] if r.get("soft") is not None else np.zeros(len(LABELS), np.float32)
                                         for r in recs]), dtype=torch.float32)
    out["soft_mask"] = torch.tensor([r.get("soft") is not None for r in recs], dtype=torch.bool)
    out["gate_t"] = torch.tensor([r["gate_t"] if r.get("gate_t") is not None else 0.5 for r in recs], dtype=torch.float32)
    out["gate_mask"] = torch.tensor([r.get("gate_t") is not None for r in recs], dtype=torch.bool)
    bir = np.array([r.get("birads", np.nan) for r in recs], dtype=float)
    out["ord_t"] = torch.tensor((bir[:, None] > np.arange(1, N_ORD + 1)[None, :]).astype(np.float32))
    out["ord_mask"] = torch.tensor(~np.isnan(bir), dtype=torch.bool)
    K = len(globals().get("FKEYS", [])) or 1
    A = np.stack([r["attr"] if r.get("attr") is not None else np.full(K, np.nan, np.float32) for r in recs]).astype(np.float32)
    out["attr_t"] = torch.tensor(np.nan_to_num(A, nan=0.0)); out["attr_mask"] = torch.tensor(~np.isnan(A), dtype=torch.bool)
    # which modalities are visible for each sample
    avail = {m: [] for m in MODALITIES}
    for r in recs:
        has = {"text": m_ok("text", r, use), "mg": m_ok("mg", r, use), "us": m_ok("us", r, use)}
        if train:
            present = [m for m in MODALITIES if has[m]]
            for m in present:
                if random.random() < MOD_DROPOUT and sum(has.values()) > 1:
                    has[m] = False
        for m in MODALITIES: avail[m].append(has[m])
    # text
    txt = [r["text"] if avail["text"][i] else None for i, r in enumerate(recs)]
    L = max([1] + [t.shape[0] for t in txt if t is not None])
    X = torch.zeros(B, L, TXT_DIM); M = torch.zeros(B, L, dtype=torch.bool)
    for i, t in enumerate(txt):
        if t is None: continue
        X[i, :len(t)] = t.float(); M[i, :len(t)] = True
    out["text"], out["text_mask"] = X, M
    # images
    for key in ("mg", "us"):
        items = [_img_tokens(r, key, train) if avail[key][i] else None for i, r in enumerate(recs)]
        N = max([1] + [it[0].shape[0] for it in items if it is not None])
        T = torch.zeros(B, N, IMG_DIM); Mk = torch.zeros(B, N, dtype=torch.bool)
        V = torch.zeros(B, N, dtype=torch.long); Pp = torch.zeros(B, N, dtype=torch.long)
        I = torch.full((B, N), -1, dtype=torch.long)
        for i, it in enumerate(items):
            if it is None: continue
            tok, view, pos, img = it; n = tok.shape[0]
            T[i, :n], V[i, :n], Pp[i, :n], I[i, :n], Mk[i, :n] = tok.float(), view, pos, img, True
        out[key], out[f"{key}_mask"], out[f"{key}_view"], out[f"{key}_pos"], out[f"{key}_img"] = T, Mk, V, Pp, I
    # token dropout (never empties a sample)
    if train and TOKEN_DROPOUT > 0:
        for key in MODALITIES:
            mk = out[f"{key}_mask"]
            drop = (torch.rand_like(mk, dtype=torch.float) < TOKEN_DROPOUT) & mk
            new = mk & ~drop
            keep_old = new.sum(1) == 0
            new[keep_old] = mk[keep_old]
            out[f"{key}_mask"] = new
    return out

def m_ok(m, r, use):
    if m not in use: return False
    if m == "text": return r["text"] is not None and len(r["text"]) > 0
    return r.get(m) is not None and len(r[m]) > 0

def to_device(b):
    return {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in b.items()}

_b = collate(make_records(labels_df.head(4)), train=True)
print({k: tuple(v.shape) for k, v in _b.items()})

## 6b) Report sections (mammography / ultrasound) and breast density (ACR)
The masked report is split at the headings *MAMMOGRAPHY* and *ULTRASONOGRAPHY / SONOMAMMOGRAPHY* (the OPINION part belongs to neither), and each section is encoded separately with Bio_ClinicalBERT. The sections are training targets only; at test time the model input is the images.

In [ ]:
SEC_CACHE = CACHE_DIR / f"sections_{cfg_hash([TEXT_MODEL, TEXT_MODE, MAX_TEXT_TOKENS, 'sec-v1'])}.pt"
_MG_HDR  = re.compile(r"\bmam+o?gr\w*")                                   # mammography / mammogrphy ...
_US_HDR  = re.compile(r"\b(?:ul\s?tra\s?so\w*|sono\w*|ultra\s?sound)")      # ultrasonography / ul trasonography / sonomammography
_OP_HDR  = re.compile(r"\b(?:opinion|impression|conclusion)s?\b")
_REP_HDR = re.compile(r"\breport\b")
_ACR     = re.compile(r"\bacr\W{0,3}([abcd])\b")

def split_report(t):
    """masked, lower-cased report -> (mammography section, ultrasound section); '' when not found."""
    if not t: return "", ""
    m = _REP_HDR.search(t); start = m.end() if (m and m.start() < 0.4 * len(t)) else 0
    op = _OP_HDR.search(t, start); end = op.start() if op else len(t)
    mg, us = _MG_HDR.search(t, start, end), _US_HDR.search(t, start, end)
    if mg and us:
        if mg.start() < us.start(): return t[mg.start():us.start()].strip(), t[us.start():end].strip()
        return t[mg.start():end].strip(), t[us.start():mg.start()].strip()
    if mg: return t[mg.start():end].strip(), ""
    if us: return "", t[us.start():end].strip()
    return "", ""

def _clean_text(pid):
    return (REPORT_TEXT.get(pid) if "REPORT_TEXT" in globals() else None) or FEATS[pid].get("clean_text", "") or ""

if SEC_CACHE.exists():
    SECTIONS = torch.load(SEC_CACHE)
else:
    SECTIONS = {}

@torch.no_grad()
def _section_entry(pid):
    mg_t, us_t = split_report(_clean_text(pid))
    return {"mg_text": mg_t, "us_text": us_t, "mg": embed_tokens(mg_t)[0].detach().cpu(), "us": embed_tokens(us_t)[0].detach().cpu()}

missing_sec = [p for p in labels_df.pid if p not in SECTIONS]
for p in tqdm(missing_sec, desc="report sections") if missing_sec else []:
    SECTIONS[p] = _section_entry(p)
if missing_sec: torch.save(SECTIONS, SEC_CACHE)

labels_df["acr"] = [(m.group(1).upper() if (m := _ACR.search(_clean_text(p))) else "NA") for p in labels_df.pid]
n = len(labels_df)
print(f"MG section found: {sum(len(SECTIONS[p]['mg']) > 0 for p in labels_df.pid)}/{n} | "
      f"US section found: {sum(len(SECTIONS[p]['us']) > 0 for p in labels_df.pid)}/{n}")
print("breast density (ACR):", labels_df.acr.value_counts().to_dict())
ex = labels_df.pid.iloc[0]
print("\nexample MG section:", SECTIONS[ex]["mg_text"][:250], "\nexample US section:", SECTIONS[ex]["us_text"][:250])

## 6c) Structured findings from the report with an LLM (training only)
The LLM reads the **masked** report (no BI-RADS) and returns true/false for 15 findings. The same findings are also extracted by a rule-based extractor, (1) as a fallback when no LLM is available and (2) to report the agreement between the LLM and the rules.

In [ ]:
import json
FINDINGS = [  # key, section used by the rule-based extractor, description given to the LLM
    ("mg_mass",            "mg",  "a mass, focal density or nodule on mammography"),
    ("mg_calc_suspicious", "mg",  "suspicious calcifications on mammography (pleomorphic, fine-linear, clustered or amorphous microcalcifications)"),
    ("mg_calc_benign",     "mg",  "benign-appearing calcifications on mammography (macro-, coarse, vascular or popcorn)"),
    ("mg_distortion",      "mg",  "architectural distortion on mammography"),
    ("mg_asymmetry",       "mg",  "focal, global or developing asymmetry on mammography"),
    ("skin_thickening",    "any", "skin thickening, skin or nipple retraction"),
    ("us_mass",            "us",  "a mass, lesion or nodule on ultrasound"),
    ("us_cyst",            "us",  "a simple or complicated cyst on ultrasound"),
    ("us_irregular",       "us",  "irregular shape or non-circumscribed (angular, spiculated, ill-defined, microlobulated) margins on ultrasound"),
    ("us_hypoechoic",      "us",  "a hypoechoic lesion on ultrasound"),
    ("us_shadowing",       "us",  "posterior acoustic shadowing on ultrasound"),
    ("us_vascular",        "us",  "internal vascularity of a lesion on ultrasound"),
    ("ln_suspicious",      "any", "suspicious, enlarged or pathological axillary lymph nodes"),
    ("post_treatment",     "any", "post-operative or post-treatment changes (scar, lumpectomy, mastectomy, clip)"),
    ("duct_ectasia",       "any", "duct ectasia or dilated ducts"),
]
FKEYS = [k for k, _, _ in FINDINGS]
_RULES = {
    "mg_mass": r"\b(mass(es)?|focal density|dense lesion|opacit\w*|nodul\w*|lesions?)\b",
    "mg_calc_suspicious": r"(pleomorphic|fine[- ]linear|linear branching|clustered|micro-?\s?calcif\w*|amorphous|suspicious calcif\w*|pathological calcif\w*)",
    "mg_calc_benign": r"(macro-?\s?calcif\w*|coarse calcif\w*|vascular calcif\w*|benign calcif\w*|popcorn)",
    "mg_distortion": r"(architectural distortion|distorted (breast )?architecture|distortion)",
    "mg_asymmetry": r"(asymmetr\w*)",
    "skin_thickening": r"(skin thicken\w*|thickened skin|nipple retraction|skin retraction)",
    "us_mass": r"\b(mass(es)?|lesions?|nodul\w*|nodules?)\b",
    "us_cyst": r"(cyst\w*)",
    "us_irregular": r"(irregular|spiculat\w*|angular|ill[- ]defined|non[- ]?circumscribed|micro-?lobulated)",
    "us_hypoechoic": r"(hypo-?\s?echoic|hypoechogen\w*)",
    "us_shadowing": r"(shadow\w*)",
    "us_vascular": r"(vasculari\w*|internal flow|doppler signal)",
    "ln_suspicious": r"(suspicious (axillary )?(lymph )?nod\w*|enlarged (axillary )?(lymph )?nod\w*|lymphadenopath\w*|pathological (axillary )?(lymph )?nod\w*|nodal enlargement)",
    "post_treatment": r"(post[- ]?operative|post[- ]?surgical|scar|lumpectomy|mastectomy|conservative surgery|post[- ]?management|post[- ]?treatment|\bclip\b)",
    "duct_ectasia": r"(duct(al)? ectasia|ductal dilat\w*|dilated ducts?|duct dilat\w*)",
}
_NEG = re.compile(r"\b(no|without|not|nor|absent|negative for|free of|neither)\b")

def rule_findings(pid):
    """Negation-aware lexicon matching per clause; None when the needed report section is missing."""
    full = _clean_text(pid); sec = SECTIONS.get(pid, {})
    out = {}
    for key, where, _ in FINDINGS:
        text = full if where == "any" else sec.get(f"{where}_text", "")
        if not text: out[key] = None; continue
        pos = False
        for clause in re.split(r"[.;:\n•]|\.\.+", text):
            for m in re.finditer(_RULES[key], clause):
                if not _NEG.search(clause[:m.start()]): pos = True
        out[key] = pos
    return out

LLM_SYS = "You are an expert breast radiologist who extracts findings from breast imaging reports. Reply with JSON only."
LLM_USER = ("Breast imaging report (BI-RADS categories were removed):\n\"\"\"\n{report}\n\"\"\"\n\n"
            "For each key, answer true if the report describes the finding as present in either breast, and false if it is "
            "absent or not mentioned. Answer null only if the report contains no {modality_note}.\n\nKeys:\n{keys}\n\n"
            "Return one JSON object with exactly these keys and boolean or null values.")

def _parse_llm(ans):
    m = re.search(r"\{.*\}", ans, re.S)
    if not m: return None
    try: d = json.loads(m.group(0))
    except Exception:
        try: d = json.loads(re.sub(r",\s*}", "}", m.group(0).replace("'", '"')))
        except Exception: return None
    conv = lambda v: (v if isinstance(v, bool) or v is None else
                      {"true": True, "yes": True, "present": True, "false": False, "no": False, "absent": False}.get(str(v).strip().lower()))
    return {k: conv(d.get(k)) for k in FKEYS}

def llm_findings(pids):
    from transformers import AutoTokenizer, AutoModelForCausalLM
    kw = {"torch_dtype": torch.float16, "device_map": "auto"}
    if LLM_4BIT:
        from transformers import BitsAndBytesConfig
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    tok = AutoTokenizer.from_pretrained(LLM_MODEL); mdl = AutoModelForCausalLM.from_pretrained(LLM_MODEL, **kw).eval()
    keys = "\n".join(f"- {k}: {d}" for k, _, d in FINDINGS)
    res, fails = {}, 0
    for pid in tqdm(pids, desc="LLM findings"):
        text = _clean_text(pid)
        if not text: res[pid] = {k: None for k in FKEYS}; continue
        msgs = [{"role": "system", "content": LLM_SYS},
                {"role": "user", "content": LLM_USER.format(report=text[:6000], keys=keys, modality_note="section for that modality")}]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(mdl.device)
        with torch.no_grad():
            gen = mdl.generate(ids, max_new_tokens=400, do_sample=False, pad_token_id=tok.eos_token_id)
        d = _parse_llm(tok.decode(gen[0, ids.shape[1]:], skip_special_tokens=True))
        if d is None: fails += 1; d = rule_findings(pid)                  # fall back to rules for this report
        res[pid] = d
    del mdl; torch.cuda.empty_cache() if torch.cuda.is_available() else None
    print(f"LLM outputs that could not be parsed (rules used instead): {fails}/{len(pids)}")
    return res

RULES = {p: rule_findings(p) for p in labels_df.pid}
ATTR_SOURCE = f"llm:{LLM_MODEL}" if USE_LLM else "rules"
F_CACHE = CACHE_DIR / f"findings_{cfg_hash([ATTR_SOURCE, TEXT_MODE, FKEYS, 'v1'])}.json"
if F_CACHE.exists():
    FIND = json.loads(F_CACHE.read_text())
else:
    FIND = RULES
    if USE_LLM:
        try:
            FIND = llm_findings(list(labels_df.pid))
        except Exception as ex:
            print(f"LLM could not be run ({type(ex).__name__}: {ex}) -> rule-based findings are used"); ATTR_SOURCE = "rules"
    F_CACHE.write_text(json.dumps(FIND))
ATTR = {p: np.array([np.nan if FIND[p].get(k) is None else float(FIND[p][k]) for k in FKEYS], np.float32) for p in labels_df.pid}

# prevalence and LLM-vs-rules agreement (report in the paper)
from sklearn.metrics import cohen_kappa_score
rows = []
for j, k in enumerate(FKEYS):
    a = np.array([ATTR[p][j] for p in labels_df.pid]); r = np.array([np.nan if RULES[p][k] is None else float(RULES[p][k]) for p in labels_df.pid])
    ok = ~np.isnan(a) & ~np.isnan(r); yv = labels_df.y.values
    rows.append({"finding": k, "available": int((~np.isnan(a)).sum()), "prevalence": np.nanmean(a),
                 "prev. BI-RADS≥4": np.nanmean(a[yv == 1]), "prev. BI-RADS≤3": np.nanmean(a[yv == 0]),
                 "agreement with rules": (a[ok] == r[ok]).mean() if ok.any() else np.nan,
                 "kappa": cohen_kappa_score(a[ok], r[ok]) if ok.sum() > 1 and len(set(a[ok])) > 1 and len(set(r[ok])) > 1 else np.nan})
FIND_TABLE = pd.DataFrame(rows); FIND_TABLE.to_csv(OUT_DIR / "findings_extraction.csv", index=False)
print("findings source:", ATTR_SOURCE); display(FIND_TABLE.round(3))

## 7) Model: MM-LDAM with a label-dependent mixture of experts
**Base (LDAM, Eqs. 1-4):** label-dependent attention over text tokens and image patches, then `z_text ⊕ z_MG ⊕ z_US → f6 → f7` (concatenation fusion).

**LD-MoE:** the experts are the per-modality heads on z_m and the joint expert f7(f6(·)). For every label l and expert e:

$$s_{e,l}=\frac{\langle W^g_e\,[u_e;\,h],\;Y_l\rangle}{\sqrt{F}}+b_e,\qquad g_{e,l}=\mathrm{softmax}_e(s_{e,l})\ \text{(present experts only)},\qquad \hat y_l=\sigma\Big(\sum_e g_{e,l}\,o_{e,l}\Big)$$

where u_e is the representation of expert e, h the joint representation of all modalities (so the router can, for example, trust US more when the mammogram is dense) and Y_l the label-name embedding of LDAM.

**Mask-aware fusion (comparison):** $f=\frac{\sum_m b_m z_m}{\max(\sum_m b_m,1)}$ followed by a linear head (Nouman & Haraguchi 2026).

**Ordinal supervision (optional candidate):** $\hat q_k=\sigma(w_k^\top h)$ for $k=1..5$ with target $\mathbb 1[\text{BI-RADS}>k]$ (training only).

**Joint expert (two options):** an MLP on the concatenation (f7∘f6 in LDAM) or the mask-aware mean of the modality representations with a linear head (`moe_avg`).

**Report-guided routing:** from the **calibrated**, out-of-fold probabilities of two section teachers, $\ell_m=\log p_m(y)$, the target is $t=\sigma((\ell_{MG}-\ell_{US})/\tau)$, and $\mathrm{BCE}\big(g_{MG}/(g_{MG}+g_{US}),\,t\big)$ is added with weight `GATE_W` (training only).

**v7:** experts = MG, US, joint concatenation and joint mask-aware mean (`moe2`). An auxiliary head on h predicts the LLM-extracted findings, $\mathcal{L}_{find}=\mathrm{BCE}(\sigma(W_f h), a)$, only for available findings and only during training.

In [ ]:
class LabelDependentAttention(nn.Module):
    """Eq. (1)-(4) of LDAM: scaled-dot cross-attention between items and label names,
    1D-CNN (n-gram) + max-pool over labels -> softmax weights beta -> weighted sum."""
    def __init__(self, d_in, F, n_labels, k1=3):
        super().__init__()
        self.f1 = nn.Linear(d_in, F)
        self.conv = nn.Conv1d(n_labels, n_labels, k1, padding=k1 // 2)
        self.F = F

    def forward(self, X, mask, Y):
        H = self.f1(X)                                            # (B, L, F)
        G = torch.einsum("blf,nf->bnl", H, Y) / math.sqrt(self.F)  # (B, NY, L)
        G = G * mask[:, None, :].float()
        u = TF.relu(self.conv(G)).max(dim=1).values                # (B, L)
        u = u.masked_fill(~mask, -1e4)
        beta = torch.softmax(u, dim=-1) * mask.float()
        beta = beta / beta.sum(-1, keepdim=True).clamp_min(1e-9)   # all-zero for samples without items
        z = torch.einsum("bl,blf->bf", beta, H)
        return z, beta


class ImageBranch(nn.Module):
    def __init__(self, d_img, d_name, F, n_labels, n_pos, E_view, ctx_layers=1, dropout=0.3):
        super().__init__()
        self.feat = nn.Sequential(nn.LayerNorm(d_img), nn.Linear(d_img, F))
        self.name = nn.Linear(d_name, F)                          # embeds the BERT view-name vector (E_S)
        self.register_buffer("E_view", E_view.clone())
        self.pos = nn.Embedding(n_pos, F)
        self.norm, self.drop = nn.LayerNorm(F), nn.Dropout(dropout)
        self.ctx = (nn.TransformerEncoder(nn.TransformerEncoderLayer(F, 4, 2 * F, dropout, batch_first=True,
                                                                     norm_first=True), ctx_layers)
                    if ctx_layers > 0 else None)
        self.att = LabelDependentAttention(F, F, n_labels, k1=1)

    def forward(self, tok, mask, view, pos, Y):
        h = self.feat(tok) + self.name(self.E_view[view]) + self.pos(pos)
        h = self.drop(self.norm(h))
        if self.ctx is not None:
            safe = mask.clone(); safe[safe.sum(1) == 0, 0] = True   # avoid all-masked rows
            h = self.ctx(h, src_key_padding_mask=~safe)
        return self.att(h, mask, Y)


class MMLDAM(nn.Module):
    def __init__(self, d_txt, d_img, E_Y, E_view, F=F_DIM, n_pos_mg=48, n_pos_us=48, fusion="concat", ordinal=False, n_attr=0):
        super().__init__()
        NY = E_Y.shape[0]
        self.register_buffer("E_Y", E_Y.clone())
        self.label_f1 = nn.Linear(E_Y.shape[1], F)                 # f1 applied to label names (shared)
        self.txt_drop = nn.Dropout(DROPOUT)
        self.text = LabelDependentAttention(d_txt, F, NY, k1=TEXT_NGRAM)
        self.mg = ImageBranch(d_img, E_view.shape[1], F, NY, n_pos_mg, E_view, CTX_LAYERS, DROPOUT)
        self.us = ImageBranch(d_img, E_view.shape[1], F, NY, n_pos_us, E_view, CTX_LAYERS, DROPOUT)
        self.missing = nn.Parameter(torch.zeros(len(MODALITIES), F))
        self.f6 = nn.Sequential(nn.Linear(len(MODALITIES) * F, F), nn.ReLU(), nn.Dropout(DROPOUT))
        self.f7 = nn.Linear(F, NY)
        self.aux = nn.ModuleDict({m: nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(F, NY)) for m in MODALITIES})
        # v2: projection heads for the image-report contrastive term (used only in the training loss)
        self.align_img = nn.Sequential(nn.Linear(F, F), nn.ReLU(), nn.Linear(F, ALIGN_DIM))
        self.align_txt = nn.Sequential(nn.Linear(d_txt, F), nn.ReLU(), nn.Linear(F, ALIGN_DIM))
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))
        # v3: label-dependent mixture of experts (created last, so concat models are identical to v2)
        self.fusion = fusion
        if fusion in ("moe", "moe_avg", "moe2"):
            n_exp = len(MODALITIES) + (2 if fusion == "moe2" else 1)                            # one expert per modality + the joint expert
            self.gate_proj = nn.ModuleList([nn.Linear(2 * F, F) for _ in range(n_exp)])   # sees [expert ; joint context]
            self.gate_bias = nn.Parameter(torch.zeros(n_exp))
        if fusion in ("avg", "moe_avg", "moe2"):                           # mask-aware average fusion (Nouman & Haraguchi 2026)
            self.avg_head = nn.Linear(F, NY)
        self.ordinal = ordinal
        if ordinal:                                                # cumulative logits P(BI-RADS > k)
            self.ord_head = nn.Linear(F, N_ORD)
        self.n_attr = n_attr
        if n_attr:                                                 # v7: head for LLM-extracted report findings
            self.attr_head = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(F, n_attr))

    def forward(self, b):
        Y = self.label_f1(self.E_Y)                                # (NY, F)
        z, beta, present = {}, {}, {}
        z["text"], beta["text"] = self.text(self.txt_drop(b["text"]), b["text_mask"], Y)
        for key, branch in (("mg", self.mg), ("us", self.us)):
            z[key], beta[key] = branch(b[key], b[f"{key}_mask"], b[f"{key}_view"], b[f"{key}_pos"], Y)
        zs = []
        for i, m in enumerate(MODALITIES):
            present[m] = b[f"{m}_mask"].any(1)
            zs.append(torch.where(present[m][:, None], z[m], self.missing[i].expand_as(z[m])))
        h = self.f6(torch.cat(zs, -1))                             # joint representation
        joint = self.f7(h)
        aux = {m: self.aux[m](z[m]) for m in MODALITIES}
        out = {"aux": aux, "beta": beta, "present": present, "label_logits": self.f7(Y), "h": h}
        if self.fusion in ("moe", "moe_avg", "moe2"):
            if self.fusion in ("moe_avg", "moe2"):                                     # mask-aware average of modalities
                Pm = torch.stack([present[m].float() for m in MODALITIES], 1)
                f_avg = (torch.stack([z[m] for m in MODALITIES], 1) * Pm[..., None]).sum(1) / Pm.sum(1, keepdim=True).clamp_min(1.0)
            if self.fusion == "moe_avg": J, UJ = [self.avg_head(f_avg)], [f_avg]
            elif self.fusion == "moe2":  J, UJ = [joint, self.avg_head(f_avg)], [h, f_avg]   # dual joint experts
            else:                        J, UJ = [joint], [h]
            U = [z[m] for m in MODALITIES] + UJ                                        # expert representations
            O = torch.stack([aux[m] for m in MODALITIES] + J, 1)                       # (B, E, NY) expert logits
            S = torch.stack([self.gate_proj[e](torch.cat([u, h], -1)) @ Y.t() for e, u in enumerate(U)], 1) / math.sqrt(Y.shape[1])
            S = S + self.gate_bias[None, :, None]
            avail = torch.stack([present[m] for m in MODALITIES] + [torch.ones_like(present[MODALITIES[0]])] * len(J), 1)
            G = torch.softmax(S.masked_fill(~avail[:, :, None], -1e4), dim=1)          # (B, E, NY), per label
            out["logits"], out["gate"] = (G * O).sum(1), G
        elif self.fusion == "avg":
            Pm = torch.stack([present[m].float() for m in MODALITIES], 1)                # (B, M)
            Zm = torch.stack([z[m] for m in MODALITIES], 1)                              # (B, M, F)
            out["logits"] = self.avg_head((Zm * Pm[..., None]).sum(1) / Pm.sum(1, keepdim=True).clamp_min(1.0))
        else:
            out["logits"] = joint
        if self.ordinal:
            out["ord_logits"] = self.ord_head(h)
        if self.n_attr:
            out["attr_logits"] = self.attr_head(h)
        if "txt_mean" in b:                                        # report embedding is a TARGET, never an input
            out["align_img"], out["align_txt"] = self.align_img(h), self.align_txt(b["txt_mean"])
        return out


def compute_loss(out, b, pos_weight, kd_w=0.0, align_w=0.0, model=None, gate_w=0.0, ord_w=0.0, attr_w=0.0):
    bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="none")
    loss = bce(out["logits"], b["target"]).mean()
    aux_l = []
    for m in MODALITIES:
        p = out["present"][m]
        if p.any():
            aux_l.append(bce(out["aux"][m][p], b["target"][p]).mean())
    if aux_l:
        loss = loss + AUX_W * torch.stack(aux_l).mean()
    NY = out["label_logits"].shape[0]
    loss = loss + LABEL_REG_W * TF.cross_entropy(out["label_logits"], torch.arange(NY, device=out["logits"].device))
    # (training only) distillation from the report teacher
    if kd_w > 0 and b["soft_mask"].any():
        sm = b["soft_mask"]
        loss = loss + kd_w * TF.binary_cross_entropy_with_logits(out["logits"][sm] / KD_T, b["soft"][sm])
    # (training only) image <-> report contrastive alignment (symmetric InfoNCE)
    if align_w > 0 and model is not None and "align_img" in out:
        am = b["txt_mean_mask"] & (out["present"]["mg"] | out["present"]["us"])
        if am.sum() >= 2:
            zi = TF.normalize(out["align_img"][am], dim=-1); zt = TF.normalize(out["align_txt"][am], dim=-1)
            sim = model.logit_scale.exp().clamp(max=100) * zi @ zt.t()
            tgt = torch.arange(sim.shape[0], device=sim.device)
            loss = loss + align_w * 0.5 * (TF.cross_entropy(sim, tgt) + TF.cross_entropy(sim.t(), tgt))
    # (training only) report-guided routing: MG-vs-US share of the gate for the BI-RADS>=4 label
    if gate_w > 0 and "gate" in out and b["gate_mask"].any():
        gm = b["gate_mask"] & out["present"]["mg"] & out["present"]["us"]
        if gm.any():
            G = out["gate"][gm][:, :, CANCER_IDX]
            i_mg, i_us = MODALITIES.index("mg"), MODALITIES.index("us")
            frac = (G[:, i_mg] / (G[:, i_mg] + G[:, i_us]).clamp_min(1e-6)).clamp(1e-4, 1 - 1e-4)
            loss = loss + gate_w * TF.binary_cross_entropy(frac, b["gate_t"][gm])
    # (training only) ordinal BI-RADS supervision
    if ord_w > 0 and "ord_logits" in out and b["ord_mask"].any():
        om = b["ord_mask"]
        loss = loss + ord_w * TF.binary_cross_entropy_with_logits(out["ord_logits"][om], b["ord_t"][om])
    # (training only) LLM-extracted report findings
    if attr_w > 0 and "attr_logits" in out and b["attr_mask"].any():
        fm = b["attr_mask"]
        loss = loss + attr_w * TF.binary_cross_entropy_with_logits(out["attr_logits"][fm], b["attr_t"][fm])
    return loss

def new_model(fusion="concat", ordinal=False, n_attr=0):
    return MMLDAM(TXT_DIM, IMG_DIM, E_Y, E_VIEW, n_pos_mg=GRID_MG[0] * GRID_MG[1],
                  n_pos_us=GRID_US[0] * GRID_US[1], fusion=fusion, ordinal=ordinal, n_attr=n_attr).to(DEVICE)

for _f, _o in (("concat", False), ("avg", False), ("moe", True), ("moe2", False)):
    _m = new_model(_f, _o, n_attr=len(FKEYS) if _f == "moe2" else 0).cpu()
    with torch.no_grad():
        _o = _m(_b)
    print(f"[{_f}{' + ordinal' if _o else ''}] trainable parameters: {sum(p.numel() for p in _m.parameters() if p.requires_grad) / 1e6:.2f} M |",
          {k: tuple(v.shape) for k, v in _o.items() if torch.is_tensor(v)})

## 8) Training and evaluation protocol
- **Outer loop:** five-fold `StratifiedGroupKFold` with group = patient; the folds are the same for all experiments, so comparisons are paired.
- **Within each fold:** `N_SEEDS` models, each with its own seed and inner-validation split (used for early stopping, Platt calibration and the Youden threshold). The final prediction is the mean of the calibrated probabilities. The test fold plays no role in any choice.
- **Teacher (for distillation):** in every outer fold the training patients are split into `TEACHER_FOLDS` parts; a teacher trained on the other parts gives the soft labels of each part (cross-fitting). Test patients are never seen by a teacher.
- The results of every experiment are saved in `outputs_mmldam/oof/`; if a run is interrupted, finished experiments are not repeated.

In [ ]:
from scipy.stats import norm, spearmanr, kruskal
from sklearn.linear_model import LogisticRegression

def batches(recs, bs, shuffle):
    idx = list(range(len(recs)))
    if shuffle: random.shuffle(idx)
    for i in range(0, len(idx), bs):
        yield [recs[j] for j in idx[i:i + bs]]

@torch.no_grad()
def predict(model, recs, use=MODALITIES, pos_weight=None, return_attn=False):
    model.eval(); probs, losses, attn, gates, hs, auxp, attrs = [], [], [], [], [], [], []
    for chunk in batches(recs, 32, False):
        b = to_device(collate(chunk, use, train=False)); out = model(b)
        probs.append(torch.sigmoid(out["logits"]).cpu()); hs.append(out["h"].cpu())
        auxp.append(torch.stack([torch.sigmoid(out["aux"][m][:, CANCER_IDX]) for m in MODALITIES], 1).cpu())
        if "gate" in out: gates.append(out["gate"].cpu())
        if "attr_logits" in out: attrs.append(torch.sigmoid(out["attr_logits"]).cpu())
        if pos_weight is not None:
            losses.append(compute_loss(out, b, pos_weight).item() * len(chunk))   # hard-label loss only
        if return_attn:
            for i in range(len(chunk)):
                attn.append({m: out["beta"][m][i][b[f"{m}_mask"][i]].cpu().numpy() for m in MODALITIES} |
                            {f"{m}_img": b[f"{m}_img"][i][b[f"{m}_mask"][i]].cpu().numpy() for m in ("mg", "us")} |
                            {f"{m}_pos": b[f"{m}_pos"][i][b[f"{m}_mask"][i]].cpu().numpy() for m in ("mg", "us")})
    P = torch.cat(probs).numpy()
    res = {"probs": P, "p_cancer": P[:, CANCER_IDX], "h": torch.cat(hs).numpy(), "aux_p": torch.cat(auxp).numpy()}
    if gates: res["gate"] = torch.cat(gates).numpy()
    if attrs: res["attr_p"] = torch.cat(attrs).numpy()
    if pos_weight is not None: res["loss"] = sum(losses) / len(recs)
    if return_attn: res["attn"] = attn
    return res

def safe_auc(y, p):
    return roc_auc_score(y, p) if len(set(y)) > 1 else float("nan")

def youden_threshold(y, p):
    if len(set(y)) < 2: return 0.5
    fpr, tpr, thr = roc_curve(y, p); return float(min(thr[np.argmax(tpr - fpr)], 1.0))

def _logit(p):
    p = np.clip(np.asarray(p, float), 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-np.asarray(x, float)))

def fit_platt(y, p):
    """Platt scaling fitted on an inner-validation set -> (a, b) or None (= identity).
    A non-positive slope is rejected so that calibration can never flip the ranking."""
    y = np.asarray(y)
    if not CALIBRATE or len(set(y)) < 2: return None
    lr = LogisticRegression(C=1.0).fit(_logit(p)[:, None], y)
    a, b = float(lr.coef_[0, 0]), float(lr.intercept_[0])
    return (a, b) if a > 0 else None

def apply_platt(params, q):
    q = np.asarray(q, float)
    if params is None: return q
    a, b = params
    return _sigmoid(a * _logit(q) + b)

def sens_at_spec(y, p, spec=SPEC_TARGET):
    fpr, tpr, _ = roc_curve(y, p); ok = fpr <= 1 - spec + 1e-12
    return float(tpr[ok].max()) if ok.any() else 0.0

def _midrank(x):
    J = np.argsort(x, kind="mergesort"); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1; i = j
    out = np.empty(N); out[J] = T
    return out

def delong_test(y, p1, p2):
    """Paired DeLong test (fast version of Sun & Xu, 2014). Returns (auc1, auc2, two-sided p)."""
    y = np.asarray(y).astype(bool); P = np.vstack([p1, p2]).astype(float)
    pos, neg = P[:, y], P[:, ~y]; m, n = pos.shape[1], neg.shape[1]
    tx = np.array([_midrank(r) for r in pos]); ty = np.array([_midrank(r) for r in neg])
    tz = np.array([_midrank(np.concatenate([a, b])) for a, b in zip(pos, neg)])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / (2.0 * n)
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    S = np.cov(v01) / m + np.cov(v10) / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    z = (aucs[0] - aucs[1]) / np.sqrt(max(var, 1e-12))
    return float(aucs[0]), float(aucs[1]), float(2 * norm.sf(abs(z)))

def paired_bootstrap_delta(y, p1, p2, n_boot=N_BOOT):
    """ΔAUC = AUC(p1) - AUC(p2) with a 95 % paired bootstrap CI."""
    y = np.asarray(y); rng = np.random.default_rng(SEED); d = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) > 1: d.append(roc_auc_score(y[i], p1[i]) - roc_auc_score(y[i], p2[i]))
    return roc_auc_score(y, p1) - roc_auc_score(y, p2), np.percentile(d, 2.5), np.percentile(d, 97.5)

def holm(pvals):
    """Holm-Bonferroni adjusted p-values."""
    p = np.asarray(pvals, float); o = np.argsort(p); m = len(p); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o):
        run = max(run, min(1.0, (m - r) * p[i])); adj[i] = run
    return adj

print("utilities ready")

In [ ]:
def train_model(tr, va, use=MODALITIES, val_use=None, seed=SEED, kd_w=0.0, align_w=0.0, verbose=False,
                fusion="concat", gate_w=0.0, ord_w=0.0, attr_w=0.0):
    """Train one model. `use` = inputs seen during training, `val_use` = inputs at validation/test time."""
    val_use = val_use or use
    seed_everything(seed)
    model = new_model(fusion, ordinal=ord_w > 0, n_attr=len(FKEYS) if attr_w > 0 else 0)
    T = np.stack([r["target"] for r in tr]); pos = T.sum(0)
    pos_weight = torch.tensor(np.clip((len(T) - pos) / np.maximum(pos, 1), 1.0, 10.0), dtype=torch.float32, device=DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * math.ceil(len(tr) / BATCH_SIZE),
                                                pct_start=0.1)
    yv = [r["y"] for r in va]
    best, best_state, wait, hist = -np.inf, None, 0, []
    for ep in range(EPOCHS):
        model.train(); tl = 0
        for chunk in batches(tr, BATCH_SIZE, True):
            b = to_device(collate(chunk, use, train=True))
            loss = compute_loss(model(b), b, pos_weight, kd_w, align_w, model, gate_w, ord_w, attr_w)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
            tl += loss.item() * len(chunk)
        pv = predict(model, va, val_use, pos_weight)
        auc = safe_auc(yv, pv["p_cancer"])
        score = auc - 1e-3 * pv["loss"] if not np.isnan(auc) else -pv["loss"]
        hist.append({"epoch": ep, "train_loss": tl / len(tr), "val_loss": pv["loss"], "val_auc": auc})
        if score > best + 1e-4:
            best, wait = score, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE: break
        if verbose and ep % 5 == 0:
            print(f"  ep {ep:3d} train {tl / len(tr):.4f} | val loss {pv['loss']:.4f} auc {auc:.3f}")
    model.load_state_dict(best_state)
    return model, pd.DataFrame(hist)


def report_records(recs, src):
    """Records whose teacher input ('text') and alignment target ('text_mean') come from `src`:
    'full' = whole masked report, 'mg_sec' / 'us_sec' = only that section of the report."""
    if src == "full": return recs
    key = src.replace("_sec", "")
    out = []
    for r in recs:
        e = SECTIONS[r["pid"]][key]
        out.append({**r, "text": e, "text_mean": e.float().mean(0) if len(e) > 0 else None})
    return out


TEACHER_CACHE, TEACHER_AUC = {}, {}

def teacher_targets(recs, y, groups, tr_i, k, df_key, src="full", outer_seed=SEED, crossfit=True):
    """Out-of-fold probabilities of a report teacher (whole report or one section) for the training patients.
    Returns (raw, cal): raw = 3-label probabilities (distillation targets),
    cal = Platt-calibrated P(BI-RADS>=4) (used for the routing targets, which compare log-likelihoods)."""
    key = (df_key, k, src, outer_seed, crossfit)
    if key in TEACHER_CACHE: return TEACHER_CACHE[key]
    raw, calp, ys, ps = {}, {}, [], []
    cf = StratifiedGroupKFold(n_splits=TEACHER_FOLDS, shuffle=True, random_state=outer_seed + 7 * k)
    splits = (list(cf.split(np.zeros(len(tr_i)), y[tr_i], groups[tr_i])) if crossfit
              else [(np.arange(len(tr_i)), np.arange(len(tr_i)))])     # in-sample: teacher scores its own training data
    for j, (a, h) in enumerate(splits):
        a_idx, h_idx = tr_i[a], tr_i[h]
        inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=outer_seed + 7 * k + j)
        t_i, v_i = next(inner.split(np.zeros(len(a_idx)), y[a_idx], groups[a_idx]))
        teacher, _ = train_model([recs[i] for i in a_idx[t_i]], [recs[i] for i in a_idx[v_i]],
                                 TEACHER_USE, seed=outer_seed + 7 * k + j)
        held = [recs[i] for i in h_idx]
        P = predict(teacher, held, TEACHER_USE)["probs"]
        for i, prob, r in zip(h_idx, P, held):
            if any(m_ok(m, r, TEACHER_USE) for m in TEACHER_USE):     # no target if the report (section) is missing
                raw[i] = prob.astype(np.float32); ys.append(r["y"]); ps.append(prob[CANCER_IDX])
        del teacher
    # calibration on the pooled out-of-fold predictions of ALL training patients (large set, no test patients)
    cal = fit_platt(ys, ps)
    calp = {i: float(apply_platt(cal, raw[i][CANCER_IDX])) for i in raw}
    TEACHER_CACHE[key], TEACHER_AUC[key] = (raw, calp), safe_auc(ys, ps)
    print(f"    teacher [{src}{'' if crossfit else ', in-sample'}] (fold {k}): AUC on training patients = {TEACHER_AUC[key]:.3f}, "
          f"targets for {len(raw)}/{len(tr_i)} patients")
    return raw, calp


def run_cv(df, train_use, test_use=None, tag="exp", kd_w=0.0, align_w=0.0, n_seeds=None, save_models=False,
           fusion="concat", report_src="full", gate_w=0.0, outer_seed=SEED, ord_w=0.0, teacher_cf=True, attr_w=0.0):
    n_seeds = n_seeds or N_SEEDS
    test_use = test_use or train_use
    base = make_records(df); recs = report_records(base, report_src)
    sec = {s: report_records(base, s) for s in ("mg_sec", "us_sec")} if gate_w > 0 else {}
    y = df.y.values; groups = df.group.values
    df_key = cfg_hash(sorted(df.pid))
    outer = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=outer_seed)
    oof = np.full(len(df), np.nan); oof_bin = np.zeros(len(df), int); fold_of = np.zeros(len(df), int)
    fold_rows = []
    for k, (tr_i, te_i) in enumerate(outer.split(np.zeros(len(y)), y, groups)):
        soft, gate_t = {}, {}
        if kd_w > 0:
            raw, _ = teacher_targets(recs, y, groups, tr_i, k, df_key, report_src, outer_seed, teacher_cf)
            soft = {i: _sigmoid(_logit(p) / KD_T).astype(np.float32) for i, p in raw.items()}
        if gate_w > 0:
            _, c_mg = teacher_targets(sec["mg_sec"], y, groups, tr_i, k, df_key, "mg_sec", outer_seed)
            _, c_us = teacher_targets(sec["us_sec"], y, groups, tr_i, k, df_key, "us_sec", outer_seed)
            for i in set(c_mg) & set(c_us):
                pm, pu = np.clip([c_mg[i], c_us[i]], 1e-4, 1 - 1e-4)          # calibrated P(BI-RADS>=4)
                lm, lu = (np.log(pm), np.log(pu)) if y[i] == 1 else (np.log(1 - pm), np.log(1 - pu))
                gate_t[i] = float(_sigmoid((lm - lu) / GATE_T))
        te = [recs[i] for i in te_i]
        val_y, val_p, test_p = [], [], []
        for s in range(n_seeds):
            sd = outer_seed + 100 * k + s
            inner = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=sd)
            a_i, v_i = next(inner.split(np.zeros(len(tr_i)), y[tr_i], groups[tr_i]))
            tr = [{**recs[i], "soft": soft.get(i), "gate_t": gate_t.get(i)} for i in tr_i[a_i]]
            va = [recs[i] for i in tr_i[v_i]]
            model, _ = train_model(tr, va, train_use, test_use, seed=sd, kd_w=kd_w, align_w=align_w,
                                   fusion=fusion, gate_w=gate_w, ord_w=ord_w, attr_w=attr_w)
            pv = predict(model, va, test_use)["p_cancer"]; pt = predict(model, te, test_use)["p_cancer"]
            yv = [r["y"] for r in va]; cal = fit_platt(yv, pv)
            val_y += yv; val_p.append(apply_platt(cal, pv)); test_p.append(apply_platt(cal, pt))
            if save_models:
                d = OUT_DIR / "models" / tag; d.mkdir(parents=True, exist_ok=True)
                torch.save({"state": {kk: v.cpu() for kk, v in model.state_dict().items()}, "platt": cal,
                            "use": test_use, "fold": k, "seed": s, "fusion": fusion, "ordinal": ord_w > 0,
                            "n_attr": len(FKEYS) if attr_w > 0 else 0},
                           d / f"fold{k}_seed{s}.pt")
            del model
        if DEVICE == "cuda": torch.cuda.empty_cache()
        thr = youden_threshold(np.array(val_y), np.concatenate(val_p))
        p = np.mean(test_p, 0)
        oof[te_i], oof_bin[te_i], fold_of[te_i] = p, (p >= thr).astype(int), k
        fold_rows.append({"fold": k, "n_test": len(te_i), "pos": int(y[te_i].sum()),
                          "AUC": safe_auc(y[te_i], p), "thr": thr,
                          "val_AUC": safe_auc(np.array(val_y), np.concatenate(val_p))})   # inner validation only
        print(f"[{tag}] fold {k}: n={len(te_i)} pos={int(y[te_i].sum())} | AUC={fold_rows[-1]['AUC']:.3f} | thr={thr:.3f}")
    res = pd.DataFrame({"pid": df.pid.values, "y": y, "p_cancer": oof, "pred": oof_bin,
                        "fold": fold_of, "birads": df.birads.values})
    return res, pd.DataFrame(fold_rows)


def exp_hash(train_use, test_use, kd_w, align_w, df, fusion="concat", report_src="full", gate_w=0.0, outer_seed=SEED,
             ord_w=0.0, teacher_cf=True, attr_w=0.0):
    items = [train_use, test_use, kd_w, align_w, N_SEEDS, CALIBRATE, KD_T, TEACHER_USE, TEACHER_FOLDS,
             ALIGN_DIM, TEXT_CACHE.name, IMG_CACHE.name, F_DIM, DROPOUT, MOD_DROPOUT, IMG_DROPOUT,
             TOKEN_DROPOUT, AUX_W, LR, EPOCHS, PATIENCE, BATCH_SIZE, N_FOLDS, SEED, sorted(df.pid)]
    if fusion != "concat" or report_src != "full" or gate_w > 0:      # v3 options (v2 experiments keep their hash)
        items += [fusion, report_src, gate_w, GATE_T, "moe-ctx-oofcal"]
    if outer_seed != SEED:
        items += ["outer", outer_seed]
    if ord_w > 0: items += ["ord", ord_w, N_ORD]                      # v4 options (earlier experiments keep their hash)
    if not teacher_cf: items += ["teacher-insample"]
    if attr_w > 0: items += ["attr", attr_w, ATTR_SOURCE, FKEYS]
    return cfg_hash(items)

def run_experiment(tag, train_use, test_use, kd_w=0.0, align_w=0.0, df=None, save_models=False,
                   fusion="concat", report_src="full", gate_w=0.0, outer_seed=SEED, ord_w=0.0, teacher_cf=True,
                   need_cols=(), attr_w=0.0):
    df = labels_df if df is None else df
    h = exp_hash(train_use, test_use, kd_w, align_w, df, fusion, report_src, gate_w, outer_seed, ord_w, teacher_cf, attr_w)
    f_oof, f_fold = OUT_DIR / "oof" / f"{tag}_{h}.csv", OUT_DIR / "oof" / f"{tag}_{h}_folds.csv"
    have_models = (not save_models) or (OUT_DIR / "models" / tag / "fold0_seed0.pt").exists()
    if f_oof.exists() and f_fold.exists() and have_models and not FORCE_RERUN:
        folds = pd.read_csv(f_fold)
        if all(c in folds.columns for c in need_cols):
            print(f"[{tag}] loaded from {f_oof.name}")
            return pd.read_csv(f_oof, dtype={"pid": str}), folds
        print(f"[{tag}] cached result lacks {need_cols} - re-running")
    t0 = time.time()
    res, folds = run_cv(df, train_use, test_use, tag, kd_w, align_w, save_models=save_models,
                        fusion=fusion, report_src=report_src, gate_w=gate_w, outer_seed=outer_seed,
                        ord_w=ord_w, teacher_cf=teacher_cf, attr_w=attr_w)
    res.to_csv(f_oof, index=False); folds.to_csv(f_fold, index=False)
    print(f"[{tag}] done in {(time.time() - t0) / 60:.1f} min | pooled AUC = {roc_auc_score(res.y, res.p_cancer):.3f}\n")
    return res, folds


def summarize(res, folds, label, test_use):
    y, p, yh = res.y.values, res.p_cancer.values, res.pred.values
    rng = np.random.default_rng(SEED); boots = []
    for _ in range(N_BOOT):
        i = rng.integers(0, len(y), len(y))
        if len(set(y[i])) > 1: boots.append(roc_auc_score(y[i], p[i]))
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0, 1]).ravel()
    return {"Method": label, "Test-time input": "+".join(MOD_NAME[m] for m in test_use),
            "AUC": roc_auc_score(y, p), "AUC lo": np.percentile(boots, 2.5), "AUC hi": np.percentile(boots, 97.5),
            "Fold AUC mean": folds.AUC.mean(), "Fold AUC sd": folds.AUC.std(ddof=1),
            "AUPRC": average_precision_score(y, p), f"Sens@{int(SPEC_TARGET * 100)}%Spec": sens_at_spec(y, p),
            "Accuracy": (tp + tn) / len(y), "Sensitivity": tp / max(tp + fn, 1), "Specificity": tn / max(tn + fp, 1),
            "Balanced acc": balanced_accuracy_score(y, yh), "F1": f1_score(y, yh, zero_division=0)}

print("training functions ready")

## 9) Experiments
| key | what |
|---|---|
| `ours` | **proposed method** (MG+US at test time) |
| `nouman` | fusion of Nouman & Haraguchi 2026, the only published MG+US work on BCMID, re-implemented on the same encoders and folds |
| `ours_norep` | same architecture without any report signal (contribution of the report and the LLM) |
| `ours_noattr` | without the LLM findings (contribution of the LLM) |
| `ours_mg`, `ours_us` | proposed method with one modality (contribution of each modality) |

Experiments run in order of priority (first `ours` and `nouman` on all splits), so the main comparison is ready if a run is interrupted. Expected time on a GPU is about 4 hours (3 splits for the primary cohort and 1 split for all patients). Results are cached, and a new run resumes where the last one stopped.

In [ ]:
IMG = ("mg", "us")
def E(tag, label, train, test, kd=0.0, align=0.0, fusion="concat", gate=0.0, attr=0.0):
    return dict(tag=tag, label=label, train=train, test=test, kd=kd, align=align, fusion=fusion, src="full",
                gate=gate, ord=0.0, cf=True, attr=attr)
OURS = dict(kd=KD_W, align=ALIGN_W, fusion="moe2", gate=GATE_W, attr=ATTR_W)
EXPERIMENTS = [                                   # in order of priority
    E("ours",        "Ours (MG+US)",                      IMG, IMG, **OURS),
    E("nouman",      "Nouman & Haraguchi (re-impl.)",     IMG, IMG, fusion="avg"),
    E("ours_norep",  "Ours w/o report supervision",       IMG, IMG, fusion="moe2"),
    E("ours_noattr", "Ours w/o LLM findings",             IMG, IMG, **{**OURS, "attr": 0.0}),
    E("ours_us",     "Ours (US only)",                    ("us",), ("us",), KD_W, ALIGN_W, fusion="moe2", attr=ATTR_W),
    E("ours_mg",     "Ours (MG only)",                    ("mg",), ("mg",), KD_W, ALIGN_W, fusion="moe2", attr=ATTR_W),
]
SPEC = {e["tag"]: e for e in EXPERIMENTS}
df_primary = labels_df[labels_df.birads != 6].reset_index(drop=True)
COHORTS = {"p": (df_primary, "_noBR6", "BI-RADS 1-5 (primary)", tuple(REPEATS_P)),
           "all": (labels_df, "", "all patients", tuple(REPEATS_ALL))}
COHORTS = {k: v for k, v in COHORTS.items() if v[3]}
for ck, (d, _, nm, seeds) in COHORTS.items():
    print(f"{nm}: {len(d)} patients, {int(d.y.sum())} positive | outer splits {seeds}")

RUNS = {ck: {e["tag"]: {} for e in EXPERIMENTS} for ck in COHORTS}
t_all = time.time()
for e in EXPERIMENTS:
    for ck, (dfc, suf, nm, seeds) in COHORTS.items():
        for rs in seeds:
            res, folds = run_experiment(f"{e['tag']}{suf}_s{rs}", e["train"], e["test"], e["kd"], e["align"], df=dfc,
                                        save_models=(e["tag"] == "ours" and rs == seeds[0]), fusion=e["fusion"],
                                        report_src="full", gate_w=e["gate"], outer_seed=rs, attr_w=e["attr"])
            RUNS[ck][e["tag"]][rs] = {**e, "res": res, "folds": folds}
print(f"all experiments: {(time.time() - t_all) / 60:.1f} min")

In [ ]:
from scipy.stats import t as tdist
METRICS = ["AUC", "AUPRC", "Accuracy", "Balanced acc", "Sensitivity", "Specificity", "F1", f"Sens@{int(SPEC_TARGET * 100)}%Spec"]
def corrected_ttest(d, n_folds=N_FOLDS):
    d = np.asarray(d, float); J = len(d); var = d.var(ddof=1) * (1.0 / J + 1.0 / (n_folds - 1))
    return float(2 * tdist.sf(abs(d.mean() / np.sqrt(max(var, 1e-12))), J - 1))

AGG, CMP = {}, {}
for ck, (dfc, suf, nm, seeds) in COHORTS.items():
    rows = []
    for e in EXPERIMENTS:
        runs = RUNS[ck][e["tag"]]
        S = pd.DataFrame([summarize(r["res"], r["folds"], e["label"], e["test"]) for r in runs.values()])
        row = {"tag": e["tag"], "Method": e["label"], "Test-time input": S["Test-time input"].iloc[0], "n splits": len(S)}
        for m in METRICS: row[m], row[m + " sd"] = S[m].mean(), (S[m].std(ddof=1) if len(S) > 1 else 0.0)
        row["AUC per split"] = " / ".join(f"{a:.3f}" for a in S["AUC"])
        rows.append(row)
    AGG[ck] = pd.DataFrame(rows).set_index("tag"); AGG[ck].to_csv(OUT_DIR / f"v7_summary_{ck}.csv")
    comps = []
    for a, b, claim in [("ours", "nouman", "ours vs. Nouman'26 fusion"), ("ours", "ours_norep", "report + LLM supervision"),
                        ("ours", "ours_noattr", "LLM findings"), ("ours", "ours_us", "MG+US vs. US"), ("ours", "ours_mg", "MG+US vs. MG")]:
        d_fold, d_pool, ps = [], [], []
        for rs in seeds:
            ra, rb = RUNS[ck][a][rs], RUNS[ck][b][rs]
            assert (ra["res"].pid.values == rb["res"].pid.values).all()
            d_fold += list(ra["folds"].AUC.values - rb["folds"].AUC.values)
            y = ra["res"].y.values; a1, a0, p = delong_test(y, ra["res"].p_cancer.values, rb["res"].p_cancer.values)
            d_pool.append(a1 - a0); ps.append(p)
        comps.append({"claim": claim, "mean ΔAUC": np.mean(d_pool), "ΔAUC per split": " / ".join(f"{x:+.3f}" for x in d_pool),
                      "splits better": f"{sum(x > 0 for x in d_pool)}/{len(d_pool)}", "folds better": f"{sum(x > 0 for x in d_fold)}/{len(d_fold)}",
                      "DeLong p per split": " / ".join(f"{x:.3f}" for x in ps),
                      "corrected p (all folds)": corrected_ttest(d_fold) if len(d_fold) > 2 else np.nan})
    CMP[ck] = pd.DataFrame(comps); CMP[ck].to_csv(OUT_DIR / f"v7_comparisons_{ck}.csv", index=False)
    print(f"\n######## {nm} — mean over {len(seeds)} outer split(s)")
    show = AGG[ck].copy()
    for m in METRICS: show[m] = [f"{v:.3f} ± {s:.3f}" for v, s in zip(AGG[ck][m], AGG[ck][m + " sd"])]
    display(show[["Method", "Test-time input"] + METRICS + ["AUC per split"]])
    display(CMP[ck].round(4))

In [ ]:
# Results figure. Every number is the same quantity as in the table: the AUC of the pooled out-of-fold
# predictions of one outer repetition, averaged over the repetitions.
plt.rcParams.update({"font.size": 7.5, "axes.titlesize": 7.8, "axes.labelsize": 7.5, "legend.fontsize": 6.3, "xtick.labelsize": 6.8,
                     "ytick.labelsize": 6.8, "axes.spines.top": False, "axes.spines.right": False, "pdf.fonttype": 42, "savefig.bbox": "tight"})
COL = {"ours": "#d62728", "nouman": "#2ca02c", "ours_norep": "#7f7f7f", "ours_noattr": "#ff7f0e", "ours_us": "#6baed6", "ours_mg": "#c7a76c"}
SHORT = {"ours_mg": "Ours\n(MG only)", "ours_us": "Ours\n(US only)", "nouman": "Nouman &\nHaraguchi", "ours_norep": "w/o report\nsupervision",
         "ours_noattr": "w/o LLM\nfindings", "ours": "Ours\n(MG+US)"}
def savefig(fig, name): fig.savefig(FIG_DIR / f"{name}.pdf"); fig.savefig(FIG_DIR / f"{name}.png", dpi=300)
GRID = np.linspace(0, 1, 201)
def split_auc(ck, t): return np.array([roc_auc_score(RUNS[ck][t][s]["res"].y, RUNS[ck][t][s]["res"].p_cancer) for s in COHORTS[ck][3]])
def mean_roc(ck, t):
    T = []
    for s in COHORTS[ck][3]:
        r = RUNS[ck][t][s]["res"]; f, tp, _ = roc_curve(r.y, r.p_cancer); T.append(np.interp(GRID, f, tp)); T[-1][0] = 0.0
    T = np.array(T); return T.mean(0), (T.std(0, ddof=1) if len(T) > 1 else np.zeros(len(GRID)))
ck0 = "p"; seeds = COHORTS[ck0][3]
fig, ax = plt.subplots(1, 2, figsize=(7.0, 2.6), gridspec_kw={"width_ratios": [1, 1.35]})
for t in ["ours_mg", "ours_us", "nouman", "ours"]:                      # (a) ROC averaged over the repetitions
    m, sd = mean_roc(ck0, t); a = split_auc(ck0, t)
    ax[0].plot(GRID, m, color=COL[t], lw=1.9 if t == "ours" else 1.0, label=f"{SPEC[t]['label']}  {a.mean():.3f}", zorder=3 if t == "ours" else 2)
    if t == "ours" and len(seeds) > 1: ax[0].fill_between(GRID, np.clip(m - sd, 0, 1), np.clip(m + sd, 0, 1), color=COL[t], alpha=0.14, lw=0)
ax[0].plot([0, 1], [0, 1], ":", color="k", lw=0.6); ax[0].set_xlabel("1 − specificity"); ax[0].set_ylabel("Sensitivity")
ax[0].set_title(f"(a) Mean ROC, BI-RADS 1–5 ({len(seeds)} repetitions)")
leg = ax[0].legend(loc="lower right", title="mean AUC", title_fontsize=6.3, frameon=True, borderpad=0.3, handlelength=1.4)
leg.get_frame().set(facecolor="white", edgecolor="none", alpha=0.92)
tags = ["ours_mg", "ours_us", "nouman", "ours_norep", "ours_noattr", "ours"]   # (b) every repetition, mean +- SD
jit = np.linspace(-0.12, 0.12, len(seeds))
for i, t in enumerate(tags):
    v = split_auc(ck0, t); sd = v.std(ddof=1) if len(v) > 1 else 0.0
    ax[1].scatter(i + jit, v, s=14, color=COL[t], zorder=3, edgecolor="white", lw=0.4)
    ax[1].errorbar(i + 0.3, v.mean(), yerr=sd, fmt="_", color="k", capsize=2, ms=8, lw=0.9)
    ax[1].text(i + 0.3, v.mean() + sd + 0.004, f"{v.mean():.3f}", ha="center", va="bottom", fontsize=6.0)
a0, a1 = split_auc(ck0, "ours_norep"), split_auc(ck0, "ours")            # same repetition without / with report supervision
for k in range(len(seeds)): ax[1].plot([tags.index("ours_norep") + jit[k], tags.index("ours") + jit[k]], [a0[k], a1[k]], color="#bbbbbb", lw=0.5, zorder=1)
ax[1].set_xticks(range(len(tags)), [SHORT[t] for t in tags]); ax[1].set_ylabel("AUC per repetition")
ax[1].set_title(f"(b) {len(seeds)} repetitions of 5-fold CV, mean ± SD")
plt.tight_layout(); savefig(fig, "v7_fig_results"); plt.show()
print("values shown in the figure (identical to the table):")
display(pd.DataFrame({SPEC[t]["label"]: [split_auc(ck0, t).mean(), split_auc(ck0, t).std(ddof=1) if len(seeds) > 1 else 0.0] for t in tags},
                     index=["mean AUC", "SD"]).T.round(3))

In [ ]:
def esc(s): return str(s).replace("&", r"\&").replace("%", r"\%")
def latex_v7(path):
    A = AGG["p"]; B = AGG.get("all"); n = A["n splits"].iloc[0]
    M = ["AUC", "AUPRC", "Accuracy", "Balanced acc", "Sensitivity", "Specificity", "F1"]
    rows = ["ours_mg", "ours_us", "nouman", "ours"]; best = {m: A.loc[rows, m].max() for m in M}
    f = lambda t, m: (r"\textbf{%.3f}" if abs(A.loc[t, m] - best[m]) < 1e-9 else "%.3f") % A.loc[t, m] + (r"$\pm$%.3f" % A.loc[t, m + " sd"] if m == "AUC" and n > 1 else "")
    L = [r"\begin{table}[t]", r"\centering",
         r"\caption{BI-RADS $\geq$4 from MG+US images on BCMID (BI-RADS 1--5, $n{=}%d$, %d positive): mean over %d repetitions of patient-grouped 5-fold CV.}"
         % (len(COHORTS["p"][0]), int(COHORTS["p"][0].y.sum()), n), r"\label{tab:main}", r"\resizebox{\columnwidth}{!}{",
         r"\begin{tabular}{l" + "c" * (len(M) + (1 if B is not None else 0)) + "}", r"\toprule",
         "Method & " + " & ".join(["AUC", "AUPRC", "Acc.", "Bal.\\,acc.", "Sens.", "Spec.", "F1"]) + (r" & AUC (all) \\" if B is not None else r" \\"), r"\midrule"]
    for t in rows:
        L.append(esc(A.loc[t, "Method"]) + " & " + " & ".join(f(t, m) for m in M) + (f" & {B.loc[t, 'AUC']:.3f}" if B is not None else "") + r" \\")
    L += [r"Nouman \& Haraguchi (reported)$^\ast$ & " + " & ".join(["--"] * len(M)) + (r" & 0.871 \\" if B is not None else r" \\"),
          r"\midrule", r"\multicolumn{%d}{l}{\textit{Ablation}} \\" % (len(M) + 1 + (1 if B is not None else 0))]
    for t in ["ours_norep", "ours_noattr"]:
        L.append(esc(A.loc[t, "Method"]) + " & " + " & ".join(f(t, m) for m in M) + (f" & {B.loc[t, 'AUC']:.3f}" if B is not None else "") + r" \\")
    L += [r"\bottomrule", r"\end{tabular}}", r"\end{table}"]
    Path(path).write_text("\n".join(L), encoding="utf-8"); return "\n".join(L)
print(latex_v7(OUT_DIR / "v7_table_main.tex"))

## 12) Interpretability of the final model (all out-of-fold)
- **Findings from images:** out-of-fold AUC of every LLM-extracted report finding, predicted by the findings head from the pixels.
- **Routing:** mean router weights for BI-RADS 1-3 versus 4-5 (Mann-Whitney test).
- **Deletion test:** does removing the most-attended patches lower the risk more than removing random patches?
- **Case cards:** true/false positives and negatives with occlusion maps (computed on the logit over 3×3 token neighbourhoods), the risk without each modality, the expert weights, and the findings predicted from the images next to what the report says.

In [ ]:
# ---- out-of-fold XAI of the proposed model (first outer split of each cohort)
def load_members(dirname):
    out = []
    for f in sorted((OUT_DIR / "models" / dirname).glob("fold*_seed*.pt")):
        ck_ = torch.load(f, map_location=DEVICE)
        m = new_model(ck_.get("fusion", "concat"), ordinal=ck_.get("ordinal", False), n_attr=ck_.get("n_attr", 0))
        m.load_state_dict(ck_["state"]); m.eval()
        out.append({"fold": ck_["fold"], "seed": ck_["seed"], "model": m, "platt": ck_["platt"], "use": ck_["use"]})
    return out
MEM, RECS = {}, {}
def members(ck, k=None):
    if ck not in MEM: MEM[ck] = load_members(f"ours{COHORTS[ck][1]}_s{COHORTS[ck][3][0]}")
    return sorted([m for m in MEM[ck] if k is None or m["fold"] == k], key=lambda m: m["seed"])
def recs_of(ck):
    if ck not in RECS: RECS[ck] = make_records(COHORTS[ck][0])
    return RECS[ck]
def final_res(ck): return RUNS[ck]["ours"][COHORTS[ck][3][0]]["res"]
I_MG, I_US = MODALITIES.index("mg"), MODALITIES.index("us"); I_CAT, I_AVG = len(MODALITIES), len(MODALITIES) + 1

def oof(ck):
    res = final_res(ck); recs = recs_of(ck); N = len(res); out = {"gate": None, "attr": None}
    for k in sorted(res.fold.unique()):
        idx = np.where(res.fold.values == k)[0]; P = [predict(m["model"], [recs[i] for i in idx], m["use"]) for m in members(ck, k)]
        for key, src in (("gate", "gate"), ("attr", "attr_p")):
            if src in P[0]:
                if out[key] is None: out[key] = np.zeros((N,) + P[0][src].shape[1:])
                out[key][idx] = np.mean([q[src] for q in P], 0)
    return out

O = oof("p"); dfp = COHORTS["p"][0]; yy = dfp.y.values
# (1) can the image model read report findings from pixels? out-of-fold AUC of each LLM finding
if O["attr"] is not None:
    rows = []
    for j, k in enumerate(FKEYS):
        a = np.array([ATTR[p][j] for p in dfp.pid]); ok = ~np.isnan(a)
        if ok.sum() > 10 and 0 < a[ok].sum() < ok.sum(): rows.append({"finding": k, "n positive": int(a[ok].sum()), "image AUC": roc_auc_score(a[ok], O["attr"][ok, j])})
    FIND_AUC = pd.DataFrame(rows).sort_values("image AUC", ascending=False); FIND_AUC.to_csv(OUT_DIR / "v7_findings_from_images.csv", index=False)
    print("report findings predicted from images (out-of-fold):"); display(FIND_AUC.round(3))
# (2) routing weights by label
if O["gate"] is not None:
    G = O["gate"][:, :, CANCER_IDX]; names = ["MG", "US", "joint (concat)", "joint (mask-aware)"]; idx_ = [I_MG, I_US, I_CAT, I_AVG]
    from scipy.stats import mannwhitneyu
    ROUTE = pd.DataFrame([{"expert": n, "BI-RADS 1-3": G[yy == 0, j].mean(), "BI-RADS 4-5": G[yy == 1, j].mean(),
                           "Mann-Whitney p": mannwhitneyu(G[yy == 0, j], G[yy == 1, j]).pvalue} for n, j in zip(names, idx_)])
    ROUTE.to_csv(OUT_DIR / "v7_routing.csv", index=False); print(ROUTE.round(4).to_string()) # display(ROUTE.round(4))
# (3) deletion test
@torch.no_grad()
def p_removed(model, rec, use, remove):
    b = to_device(collate([rec], use, train=False))
    for key, idx in remove.items():
        if len(idx): b[f"{key}_mask"][0, torch.as_tensor(np.asarray(idx), dtype=torch.long, device=DEVICE)] = False
    model.eval(); return torch.sigmoid(model(b)["logits"])[0, CANCER_IDX].item()
res = final_res("p"); rng = np.random.default_rng(SEED); top_d, rnd_d = [], []
for i in np.where(res.y.values == 1)[0]:
    m = members("p", int(res.fold.iloc[i]))[0]; rec = recs_of("p")[i]; use = m["use"]
    a = predict(m["model"], [rec], use, return_attn=True)["attn"][0]
    p0 = p_removed(m["model"], rec, use, {})
    if p0 < 1e-3: continue
    top = {k: np.argsort(-a[k])[:max(1, int(0.2 * len(a[k])))] for k in ("mg", "us") if len(a[k])}
    rnd = {k: rng.choice(len(a[k]), len(v), replace=False) for k, v in top.items()}
    top_d.append(1 - p_removed(m["model"], rec, use, top) / p0); rnd_d.append(1 - p_removed(m["model"], rec, use, rnd) / p0)
from scipy.stats import wilcoxon
print(f"deletion of the 20% most-attended patches: relative drop {np.mean(top_d):.3f} vs. random {np.mean(rnd_d):.3f} "
      f"(Wilcoxon p = {wilcoxon(top_d, rnd_d).pvalue:.2e}, n = {len(top_d)})")
# (4) case cards: out-of-fold examples (true/false positive/negative)
# For each case: the most influential MG view and US image with an occlusion map (drop of the logit of the malignant
# label when a 3x3 token neighbourhood is removed), the risk without each modality, the expert weights, and the
# findings predicted from the images next to what the report says. All risks are the calibrated ensemble, as in the table.
from scipy.ndimage import gaussian_filter
from matplotlib.patches import Rectangle, Patch
from matplotlib.lines import Line2D
CARD_CASES = ("TP", "FP", "TN", "FN")      # e.g. ("TP", "FP") for a single-row figure
CARD_NAMES = {"TP": "True positive", "FP": "False positive", "TN": "True negative", "FN": "False negative"}
EXCLUDE_FINDINGS = ("post_treatment",)     # not a lesion finding; left off the cards
OCCL_MIN = 0.05                            # smallest logit drop that is drawn
EXP_COLORS = ["#2f6db5", "#1a9e77", "#7b5ea7", "#b9aedc"]
EXP_NAMES = ["MG expert", "US expert", "joint (concat.)", "joint (mean)"]
NICE = {"mg_mass": "MG mass", "mg_calc_suspicious": "MG suspicious calcification", "mg_calc_benign": "MG benign calcification",
        "mg_distortion": "MG architectural distortion", "mg_asymmetry": "MG asymmetry", "skin_thickening": "skin thickening",
        "us_mass": "US mass", "us_cyst": "US cyst", "us_irregular": "US irregular margin", "us_hypoechoic": "US hypoechoic lesion",
        "us_shadowing": "US posterior shadowing", "us_vascular": "US internal vascularity", "ln_suspicious": "suspicious lymph node",
        "post_treatment": "post-treatment change", "duct_ectasia": "duct ectasia"}

@torch.no_grad()
def _case_logit(model, rec, use, remove=None):
    b = to_device(collate([rec], use, train=False))
    for key, idx in (remove or {}).items():
        if len(idx): b[f"{key}_mask"][0, torch.as_tensor(np.asarray(idx), dtype=torch.long, device=DEVICE)] = False
    model.eval(); return float(model(b)["logits"][0, CANCER_IDX].item())
def _risk(mem, logits):
    return float(np.mean([apply_platt(m["platt"], np.atleast_1d(1.0 / (1.0 + np.exp(-z))))[0] for m, z in zip(mem, logits)]))

def case_data(ck, i):
    res = final_res(ck); k = int(res.fold.iloc[i]); mem = members(ck, k); rec = recs_of(ck)[i]; use = mem[0]["use"]
    F = RUNS[ck]["ours"][COHORTS[ck][3][0]]["folds"]; pid = res.pid.iloc[i]
    full = [_case_logit(m["model"], rec, use) for m in mem]
    P = [predict(m["model"], [rec], use, return_attn=True) for m in mem]; a = P[0]["attn"][0]
    D = {"pid": pid, "y": int(res.y.iloc[i]), "pred": int(res.pred.iloc[i]), "p": float(res.p_cancer.iloc[i]),
         "birads": int(COHORTS[ck][0].birads.iloc[i]), "thr": float(F.loc[F.fold == k, "thr"].iloc[0]),
         "gate": np.mean([q["gate"][0, :, CANCER_IDX] for q in P], 0)[[I_MG, I_US, I_CAT, I_AVG]] if "gate" in P[0] else None,
         "attr": np.mean([q["attr_p"][0] for q in P], 0) if "attr_p" in P[0] else None,
         "report": ATTR.get(pid) if "ATTR" in globals() else None, "maps": {}, "without": {}}
    for key, files, grid in (("mg", rec["mg_files"], GRID_MG), ("us", rec["us_files"], GRID_US)):
        if key not in use or len(a[key]) == 0: continue
        others = tuple(u for u in use if u != key)
        D["without"][key] = _risk(mem, [_case_logit(m["model"], rec, others) for m in mem]) if others else np.nan
        img, pos = a[f"{key}_img"], a[f"{key}_pos"]
        drop_img = [np.mean([f - _case_logit(m["model"], rec, use, {key: np.where(img == j)[0]}) for m, f in zip(mem, full)]) for j in range(len(files))]
        j = int(np.argmax(drop_img)); sel = np.where(img == j)[0]; rr, cc = np.divmod(pos[sel], grid[1])
        hm = np.zeros(grid[0] * grid[1])
        for t_, r0, c0 in zip(sel, rr, cc):
            nb = sel[(np.abs(rr - r0) <= 1) & (np.abs(cc - c0) <= 1)]
            hm[pos[t_]] = np.mean([f - _case_logit(m["model"], rec, use, {key: nb}) for m, f in zip(mem, full)])
        D["maps"][key] = (files[j], hm.reshape(grid))
    return D

def _overlay(ax, im, hm):
    up = np.asarray(Image.fromarray(np.clip(hm, 0, None).astype(np.float32)).resize(im.shape[::-1], Image.BICUBIC))
    up = gaussian_filter(up, 0.25 * min(im.shape) / max(hm.shape)) * (gaussian_filter((im > 0.04).astype(float), 3) > 0.5)
    ax.imshow(im, cmap="gray", vmin=0, vmax=1)
    if up.max() > OCCL_MIN:
        u = up / up.max(); rgba = plt.get_cmap("inferno")(u); rgba[..., 3] = np.where(u > 0.25, 0.55 * u, 0.0)
        ax.imshow(rgba); ax.contour(u, levels=[0.6], colors=["#ffd84d"], linewidths=0.6)
    ax.set_xticks([]); ax.set_yticks([]); [s.set_linewidth(0.4) for s in ax.spines.values()]

def _report_status(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "not stated", "#777777"
    return ("present", "#1b7837") if v >= 0.5 else ("absent", "#b2182b")

def case_cards(cards, name, ncols=2):
    nrows = int(np.ceil(len(cards) / ncols)); H = 2.3 * nrows + 0.3
    fig = plt.figure(figsize=(7.0, H))
    outer = fig.add_gridspec(nrows, ncols, left=0.01, right=0.99, top=1 - 0.3 / H, bottom=0.01, wspace=0.12, hspace=0.14)
    for c, (code, D) in enumerate(cards):
        g = outer[c // ncols, c % ncols].subgridspec(2, 2, height_ratios=[1.3, 1.0], width_ratios=[0.72, 1.28], hspace=0.10, wspace=0.04)
        for col, key in enumerate(("mg", "us")):
            ax = fig.add_subplot(g[0, col])
            if key in D["maps"]:
                f, hm = D["maps"][key]; _overlay(ax, (preprocess_mammogram if key == "mg" else preprocess_ultrasound)(f), hm)
                w = D["without"].get(key, np.nan)
                ax.set_title(f"{key.upper()}   risk without {key.upper()}: {w:.2f}" if np.isfinite(w) else key.upper(), fontsize=6.2, pad=2)
            else:
                ax.axis("off"); ax.text(0.5, 0.5, f"no {key.upper()}", ha="center", va="center", fontsize=6.5, transform=ax.transAxes)
        ax = fig.add_subplot(g[1, :]); ax.axis("off"); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ok = D["y"] == D["pred"]
        ax.text(0.0, 0.99, f"{CARD_NAMES[code]}  ·  BI-RADS {D['birads']}", fontsize=7.2, weight="bold", color="#1b7837" if ok else "#b2182b", va="top")
        X0, XW = 0.30, 0.42
        ax.text(0.0, 0.76, "P(BI-RADS ≥ 4)", fontsize=6.2, va="center")
        ax.add_patch(Rectangle((X0, 0.71), XW, 0.10, fc="#eeeeee", lw=0)); ax.add_patch(Rectangle((X0, 0.71), XW * D["p"], 0.10, fc="#c0392b", lw=0))
        ax.plot([X0 + XW * D["thr"]] * 2, [0.66, 0.86], color="k", lw=0.9)
        ax.text(X0 + XW + 0.02, 0.76, f"{D['p']:.2f}  (threshold {D['thr']:.2f})", fontsize=6.0, va="center")
        if D["gate"] is not None:
            gw = D["gate"] / D["gate"].sum(); x = X0
            ax.text(0.0, 0.55, "expert weights", fontsize=6.2, va="center")
            for w_, c_ in zip(gw, EXP_COLORS): ax.add_patch(Rectangle((x, 0.50), XW * w_, 0.10, fc=c_, lw=0)); x += XW * w_
            ax.text(X0 + XW + 0.02, 0.55, f"MG {gw[0]:.2f}  US {gw[1]:.2f}", fontsize=6.0, va="center")
        if D["attr"] is not None:
            ax.text(0.0, 0.34, "findings predicted from the images", fontsize=6.2, va="center")
            ax.text(0.62, 0.34, "probability", fontsize=6.0, va="center", color="#555555"); ax.text(0.80, 0.34, "in the report", fontsize=6.0, va="center", color="#555555")
            idx = [j for j in np.argsort(-D["attr"]) if FKEYS[j] not in EXCLUDE_FINDINGS][:3]
            for r_, j in enumerate(idx):
                y_ = 0.20 - r_ * 0.12; st, sc = _report_status(None if D["report"] is None else float(D["report"][j]))
                ax.text(0.02, y_, NICE.get(FKEYS[j], FKEYS[j].replace("_", " ")), fontsize=6.0, va="center")
                ax.text(0.62, y_, f"{D['attr'][j]:.2f}", fontsize=6.0, va="center"); ax.text(0.80, y_, st, fontsize=6.0, va="center", color=sc)
    handles = [Patch(color=c_, label=n_) for c_, n_ in zip(EXP_COLORS, EXP_NAMES)] + \
              [Patch(color=plt.get_cmap("inferno")(0.75), label="region whose removal lowers the risk"),
               Line2D([0], [0], color="k", marker="|", ls="none", ms=7, label="decision threshold")]
    fig.legend(handles=handles, loc="upper center", ncol=6, frameon=False, fontsize=6.2, handlelength=1.0, bbox_to_anchor=(0.5, 1.0))
    savefig(fig, name); plt.show()

r = final_res("p"); dfp_ = COHORTS["p"][0]
SELECT = {"TP": r[(r.y == 1) & (r.pred == 1)].sort_values("p_cancer", ascending=False),
          "FP": r[(r.y == 0) & (r.pred == 1)].sort_values("p_cancer", ascending=False),
          "TN": r[(r.y == 0) & (r.pred == 0)].sort_values("p_cancer", ascending=True),
          "FN": r[(r.y == 1) & (r.pred == 0)].sort_values("p_cancer", ascending=True)}
cards = []
for code in CARD_CASES:
    if len(SELECT[code]) == 0: print(f"no {CARD_NAMES[code].lower()} in the out-of-fold predictions"); continue
    pid = SELECT[code].pid.iloc[0]; i = int(np.where(dfp_.pid.values == pid)[0][0])
    cards.append((code, case_data("p", i))); print(f"{CARD_NAMES[code]}: {pid}  P = {cards[-1][1]['p']:.2f}")
case_cards(cards, "v7_fig_cases")

## What to send back
1. LLM findings table (`findings_extraction.csv`): prevalence of the findings and agreement of the LLM with the rules
2. The results tables (`v7_summary_p.csv`, `v7_comparisons_p.csv`, and the same two for `all`)
3. XAI outputs: findings predicted from the pixels, routing weights and the patch-deletion test
4. The figures `v7_fig_results` and `v7_fig_cases`, and `v7_table_main.tex`